In [1]:
import copy
import hashlib
import random
import warnings
from pathlib import Path
from sklearn.decomposition import PCA

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from mordred import Calculator
from mordred import (
    ABCIndex,
    AcidBase,
    Aromatic,
    BCUT,
    BondCount,
    CarbonTypes,
    Chi,
    Constitutional,
    CPSA,
    EState,
    EccentricConnectivityIndex,
    HydrogenBond,
    KappaShapeIndex,
    Lipinski,
    LogS,
    McGowanVolume,
    MoeType,
    Polarizability,
    RingCount,
    RotatableBond,
    SLogP,
    TopoPSA,
    TopologicalCharge,
    TopologicalIndex,
    VdwVolumeABC,
    Weight,
    ZagrebIndex,
)
from rdkit import Chem
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GroupShuffleSplit
from torch.utils.data import DataLoader, TensorDataset


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42

DESCRIPTOR_CLASSES = [
    SLogP,
    TopoPSA,
    HydrogenBond,
    McGowanVolume,
    Polarizability,
    Weight,
    LogS,
    Lipinski,
    MoeType,
    CPSA,
    RotatableBond,
    RingCount,
    Aromatic,
    KappaShapeIndex,
    Constitutional,
    EState,
    BCUT,
    VdwVolumeABC,
    TopologicalIndex,
    ZagrebIndex,
    ABCIndex,
    AcidBase,
    BondCount,
    CarbonTypes,
    Chi,
    EccentricConnectivityIndex,
    TopologicalCharge,
]


def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def make_calculator():
    return Calculator(DESCRIPTOR_CLASSES, ignore_3D=True)


def smiles_to_mordred(smiles, calculator):
    """Return finite descriptors; Mordred failures are represented as NaN."""
    n_desc = len(calculator.descriptors)
    if pd.isna(smiles) or not str(smiles).strip():
        return np.full(n_desc, np.nan, dtype=np.float64)
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return np.full(n_desc, np.nan, dtype=np.float64)

    values = []
    for value in calculator(mol):
        try:
            value = float(value)
        except (TypeError, ValueError, OverflowError):
            value = np.nan
        values.append(value if np.isfinite(value) else np.nan)
    return np.asarray(values, dtype=np.float64)

def compact_descriptors_unique_solvents(df, fp1_raw, fp2_raw, idx_train, variance=0.95):
    """Fits PCA strictly on the UNIQUE solvents present in the training set."""
    # 1. Collect unique SMILES from the training set
    train_df = df.iloc[idx_train]
    unique_smiles_train = pd.unique(
        train_df[["SMILES_1", "SMILES_2"]].values.ravel()
    )

    # 2. Map SMILES to their raw Mordred descriptor vectors
    # (Extracting 1 copy of descriptors per unique training solvent)
    smiles_to_fp = {}
    for idx in idx_train:
        s1 = df.iloc[idx]["SMILES_1"]
        s2 = df.iloc[idx]["SMILES_2"]
        if s1 not in smiles_to_fp:
            smiles_to_fp[s1] = fp1_raw[idx]
        if s2 not in smiles_to_fp:
            smiles_to_fp[s2] = fp2_raw[idx]

    # Matrix X: Rows = unique training solvents, Columns = Mordred descriptors
    X_unique_train = np.vstack([smiles_to_fp[smi] for smi in unique_smiles_train])

    # 3. Fit Standardizer strictly on unique training solvents
    standardizer = TrainOnlyStandardizer().fit([X_unique_train])
    X_unique_std = standardizer.transform(X_unique_train)

    # 4. Fit ONE shared PCA on unique standardized solvents
    pca = PCA(n_components=variance, svd_solver="full", random_state=SEED)
    pca.fit(X_unique_std)

    # 5. Transform full dataset fp1 and fp2 using the fitted standardizer and PCA
    fp1_std = standardizer.transform(fp1_raw)
    fp2_std = standardizer.transform(fp2_raw)

    fp1_reduced = pca.transform(fp1_std)
    fp2_reduced = pca.transform(fp2_std)

    return fp1_reduced, fp2_reduced, standardizer, pca


class TrainOnlyStandardizer:
    """Drop columns with any NaN in training data, remove train-constant columns, then standardize."""

    def fit(self, arrays):
        stacked = np.vstack(arrays).astype(np.float64)
        finite = np.isfinite(stacked)

        # Drop any column that has at least one non-finite (NaN/inf) value in training data
        no_missing = finite.all(axis=0)

        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            self.mean = np.nanmean(stacked, axis=0)
            self.scale = np.nanstd(stacked, axis=0)

        non_constant = self.scale > 1e-12
        self.keep = no_missing & non_constant

        self.mean = np.where(self.keep, self.mean, 0.0)
        self.scale = np.where(self.keep, self.scale, 1.0)

        n_total = stacked.shape[1]
        n_nan_dropped = int((~no_missing).sum())
        n_constant_dropped = int((no_missing & ~non_constant).sum())
        n_remaining = int(self.keep.sum())
        print(
            f"[TrainOnlyStandardizer] Descriptors: {n_total} total | "
            f"{n_nan_dropped} dropped (NaN) | "
            f"{n_constant_dropped} dropped (constant) | "
            f"{n_remaining} remaining"
        )

        return self

    def transform(self, array):
        array = np.asarray(array, dtype=np.float64)
        filled = np.where(np.isfinite(array), array, self.mean)
        return ((filled - self.mean) / self.scale)[:, self.keep]

    @property
    def n_features_out(self):
        return int(self.keep.sum())


def get_canonical_smiles(smiles_str):
    """Convert SMILES string to canonical format for reliable matching."""
    try:
        mol = Chem.MolFromSmiles(str(smiles_str))
        return Chem.MolToSmiles(mol) if mol else smiles_str
    except Exception:
        return smiles_str

   
def group_split(df, SMILES__1, SMILES__2, seed=SEED, val_size=0.15):
    """
    1. Holds out solvent mixture points for the test set.
    2. Uses GroupShuffleSplit on remaining data to prevent train/val mixture leakage.
    """
    target_1 = get_canonical_smiles(SMILES__1)
    target_2 = get_canonical_smiles(SMILES__2)

    s1_canon = df["SMILES_1"].apply(get_canonical_smiles)
    s2_canon = df["SMILES_2"].apply(get_canonical_smiles)

    # Identify holdout test pair, excluding pure solvents)
    is_pair_1 = (s1_canon == target_1) & (s2_canon == target_2)
    is_pair_2 = (s1_canon == target_2) & (s2_canon == target_1)

    mask_test_pair = is_pair_1 | is_pair_2
    mask_pure_solvent = (df["comp_1"] == 0) | (df["comp_1"] == 100) | (df["comp_1"] == 1.0)
    mask_solv_test = mask_test_pair & (~mask_pure_solvent)

    # Split dataset into test and remaining subsets
    df_test = df[mask_solv_test]
    df_remaining = df[~mask_solv_test]

    idx_test = df_test.index.values

    # Create symmetric mixture group keys for df_remaining to prevent train/val leakage
    # Sorting SMILES pair handles order variations (e.g., A+B vs B+A)
    smiles_sorted = np.sort(df_remaining[["SMILES_1", "SMILES_2"]].values, axis=1)
    groups = (
        smiles_sorted[:, 0] + "|" + 
        smiles_sorted[:, 1] + "|" + 
        df_remaining["comp_1"].astype(str)
    )

    # GroupShuffleSplit for Train/Val
    gss = GroupShuffleSplit(n_splits=1, test_size=val_size, random_state=seed)
    train_idx_rel, val_idx_rel = next(gss.split(df_remaining, groups=groups))

    # Map relative position indices back to original dataframe index labels
    idx_train = df_remaining.index[train_idx_rel].values
    idx_val = df_remaining.index[val_idx_rel].values

    return idx_train, idx_val, idx_test


def random_row_split(df, seed=SEED):
    """Reproduce the original row-level 70/15/15 random split."""
    all_indices = np.arange(len(df))
    idx_train, idx_temp = train_test_split(
        all_indices, test_size=0.30, random_state=seed
    )
    idx_val, idx_test = train_test_split(
        idx_temp, test_size=0.50, random_state=seed
    )
    group_columns = ["SMILES_1", "SMILES_2", "comp_1", "comp_2"]
    group_keys = df[group_columns].astype(str).agg("|".join, axis=1).to_numpy()
    signature = hashlib.md5(
        np.concatenate([idx_train, idx_val, idx_test]).tobytes()
    ).hexdigest()
    return idx_train, idx_val, idx_test, signature, group_keys

def temperature_holdout_split(df, mode="extrapolation"):
    if mode not in {"interpolation", "extrapolation"}:
        raise ValueError("mode must be 'interpolation' or 'extrapolation'")

    group_columns = ["SMILES_1", "SMILES_2", "comp_1", "comp_2"]
    group_keys = df[group_columns].astype(str).agg("|".join, axis=1).to_numpy()
    temperatures = df["Temperature (K)"].to_numpy(float)
    train, val, test = [], [], []

    for group in np.unique(group_keys):
        indices = np.flatnonzero(group_keys == group)
        order = indices[np.argsort(temperatures[indices], kind="stable")]
        n = len(order)
        if n < 6:
            train.extend(order.tolist())
            continue

        if mode == "extrapolation":
            test_positions = {0, n - 1}
            val_positions = {1, n - 2}
        else:
            test_positions = {n // 2}
            val_positions = {n // 2 - 1}

        for position, index in enumerate(order):
            if position in test_positions:
                test.append(int(index))
            elif position in val_positions:
                val.append(int(index))
            else:
                train.append(int(index))

    idx_train = np.sort(np.asarray(train, dtype=int))
    idx_val = np.sort(np.asarray(val, dtype=int))
    idx_test = np.sort(np.asarray(test, dtype=int))
    signature = hashlib.md5(
        np.concatenate([idx_train, idx_val, idx_test]).tobytes()
    ).hexdigest()
    return idx_train, idx_val, idx_test, signature, group_keys


import numpy as np
import hashlib
"""
def composition_holdout_split(df, mode="extrapolation"):
    if mode not in {"interpolation", "extrapolation"}:
        raise ValueError("mode must be 'interpolation' or 'extrapolation'")

    # Group by chemical pair only, allowing composition to vary within groups
    group_columns = ["SMILES_1", "SMILES_2"]
    group_keys = df[group_columns].astype(str).agg("|".join, axis=1).to_numpy()
    
    # Sort and evaluate based on composition (e.g., mole fraction comp_1)
    compositions = df["comp_1"].to_numpy(float)
    train, val, test = [], [], []

    for group in np.unique(group_keys):
        indices = np.flatnonzero(group_keys == group)
        order = indices[np.argsort(compositions[indices], kind="stable")]
        n = len(order)
        
        # Fallback for systems with too few composition points
        if n < 6:
            train.extend(order.tolist())
            continue

        if mode == "extrapolation":
            # Hold out extreme compositions (e.g., pure components or dilute limits)
            test_positions = {0, 1, 2, 3, 4, 5, n - 5, n - 4, n - 3, n - 2, n - 1, n}
            val_positions = {6, 7, 8, 9, 10, 11, n-11, n-10, n-9, n-8, n-7, n-6}
        else:
            # Hold out intermediate compositions
            test_positions = {n // 2, n // 2 + 1, n // 2 - 1, n // 2 + 2, n // 2 - 2, n // 2 + 3, n // 2 - 3, n // 2 + 4, n // 2 - 4, n // 2 + 5, n // 2 - 5}
            val_positions = {n // 2 - 6, n // 2 + 6, n // 2 - 7, n // 2 + 7, n // 2 - 8, n // 2 + 8, n // 2 - 9, n // 2 + 9, n // 2 - 10, n // 2 + 10, n // 2 - 11}

        for position, index in enumerate(order):
            if position in test_positions:
                test.append(int(index))
            elif position in val_positions:
                val.append(int(index))
            else:
                train.append(int(index))

    idx_train = np.sort(np.asarray(train, dtype=int))
    idx_val = np.sort(np.asarray(val, dtype=int))
    idx_test = np.sort(np.asarray(test, dtype=int))
    
    signature = hashlib.md5(
        np.concatenate([idx_train, idx_val, idx_test]).tobytes()
    ).hexdigest()
    
    return idx_train, idx_val, idx_test, signature, group_keys
"""
def composition_holdout_split(df, mode="extrapolation"):
    if mode not in {"interpolation", "extrapolation"}:
        raise ValueError("mode must be 'interpolation' or 'extrapolation'")

    group_columns = ["SMILES_1", "SMILES_2"]
    group_keys = df[group_columns].astype(str).agg("|".join, axis=1).to_numpy()
    compositions = df["comp_1"].to_numpy(float)

    train, val, test = [], [], []

    for group in np.unique(group_keys):
        indices = np.flatnonzero(group_keys == group)
        
        # Get unique composition values for this system, sorted numerically
        group_comps = compositions[indices]
        unique_comps = np.sort(np.unique(group_comps))
        n_comps = len(unique_comps)

        # Fallback if the system doesn't have enough distinct composition points
        if n_comps < 5:
            train.extend(indices.tolist())
            continue

        # Map each unique composition value to a train/val/test split
        if mode == "extrapolation":
            # Hold out lowest/highest compositions for Test, next tier for Val
            test_comps = {unique_comps[0], unique_comps[-1]}
            val_comps = {unique_comps[1], unique_comps[-2]} if n_comps >= 4 else set()
        else:
            # Interpolation: Hold out middle compositions for Test, adjacent for Val
            mid = n_comps // 2
            test_comps = {unique_comps[mid]}
            val_comps = {unique_comps[mid - 1], unique_comps[mid + 1]} if n_comps >= 3 else set()

        for idx in indices:
            c = compositions[idx]
            if c in test_comps:
                test.append(idx)
            elif c in val_comps:
                val.append(idx)
            else:
                train.append(idx)

    idx_train = np.sort(np.asarray(train, dtype=int))
    idx_val = np.sort(np.asarray(val, dtype=int))
    idx_test = np.sort(np.asarray(test, dtype=int))

    signature = hashlib.md5(
        np.concatenate([idx_train, idx_val, idx_test]).tobytes()
    ).hexdigest()

    return idx_train, idx_val, idx_test, signature, group_keys

def load_raw_data(data_path):
    data_path = Path(data_path)
    df = pd.read_csv(data_path)
    required = {
        "LogS",
        "Temperature (K)",
        "SMILES_1",
        "SMILES_2",
        "comp_1",
        "comp_2",
    }
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")
    numeric = df[["LogS", "Temperature (K)", "comp_1", "comp_2"]].to_numpy(float)
    if not np.isfinite(numeric).all():
        raise ValueError("LogS, temperature, and composition must be finite")
    if (numeric[:, 1] <= 0).any():
        raise ValueError("Temperature (K) must be positive")
    total = numeric[:, 2] + numeric[:, 3]
    if (total <= 0).any():
        raise ValueError("Composition totals must be positive")
    x1 = (numeric[:, 2] / total).reshape(-1, 1)
    if ((x1 < 0) | (x1 > 1)).any():
        raise ValueError("Normalized composition must lie in [0, 1]")
    return df, x1, numeric[:, 1:2], numeric[:, 0:1]


def compute_fingerprints(df, calculator):
    """Compute each unique SMILES once instead of once per data row."""
    unique_smiles = pd.unique(pd.concat([df["SMILES_1"], df["SMILES_2"]]))
    lookup = {
        smiles: smiles_to_mordred(smiles, calculator) for smiles in unique_smiles
    }
    fp1 = np.vstack([lookup[s] for s in df["SMILES_1"]])
    fp2 = np.vstack([lookup[s] for s in df["SMILES_2"]])
    return fp1, fp2, lookup


def make_loaders(arrays_train, arrays_val, arrays_test, batch_size=32):
    train_t = [torch.tensor(a, dtype=torch.float32) for a in arrays_train]
    val_t = [torch.tensor(a, dtype=torch.float32) for a in arrays_val]
    test_t = [torch.tensor(a, dtype=torch.float32) for a in arrays_test]
    return (
        DataLoader(TensorDataset(*train_t), batch_size=batch_size, shuffle=True),
        DataLoader(TensorDataset(*val_t), batch_size=len(val_t[0]), shuffle=False),
        DataLoader(TensorDataset(*test_t), batch_size=len(test_t[0]), shuffle=False),
    )


ACTIVATIONS = {"Tanh": nn.Tanh, "ReLU": nn.ReLU, "Sigmoid": nn.Sigmoid, "ELU": nn.ELU}


def initialize_weights(module):
    if isinstance(module, nn.Linear):
        nn.init.xavier_uniform_(module.weight)
        nn.init.constant_(module.bias, 0.01)


def generate_mlp(n_input, hidden_layers, n_out, dropout, activation):
    """Dropout is used only between hidden layers, never on model outputs."""
    sizes = [n_input] + list(hidden_layers) + [n_out]
    layers = []
    for index, (left, right) in enumerate(zip(sizes[:-1], sizes[1:])):
        layers.append(nn.Linear(left, right))
        if index < len(sizes) - 2:
            layers.extend([ACTIVATIONS[activation](), nn.Dropout(dropout)])
    return nn.Sequential(*layers)


class WeightedSumFingerprintModel(nn.Module):
    """Mix compact solvent descriptors directly before the neural backbone."""

    def __init__(self, n_desc):
        super().__init__()
        self.fp_branch = generate_mlp(n_desc, [64, 32], 16, 0.1, "Tanh")
        self.physpar_head = generate_mlp(16, [16], 2, 0.1, "Tanh")
        self.A_min, self.A_max = 2.5, 6.5
        self.B_min, self.B_max = -2500.0, -500.0
        self.apply(initialize_weights)

    def get_AB(self, fp1, fp2, x1):
        x2 = 1.0 - x1
        mixed_descriptors = x1 * fp1 + x2 * fp2
        raw = self.physpar_head(self.fp_branch(mixed_descriptors))
        A = torch.sigmoid(raw[:, 0:1]) * (self.A_max - self.A_min) + self.A_min
        B = torch.sigmoid(raw[:, 1:2]) * (self.B_max - self.B_min) + self.B_min
        return A, B

    def forward(self, fp1, fp2, x1, temperature):
        A, B = self.get_AB(fp1, fp2, x1)
        return A + B / temperature

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


def train_model(model, train_loader, val_loader, forward_batch, max_epochs=600, patience=50):
    model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    criterion = nn.MSELoss()
    best_loss, best_state, wait = float("inf"), None, 0
    train_losses, val_losses = [], []
    for epoch in range(1, max_epochs + 1):
        model.train()
        total_loss = 0.0
        for batch in train_loader:
            batch = [x.to(DEVICE) for x in batch]
            optimizer.zero_grad()
            loss = criterion(forward_batch(model, batch), batch[-1])
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(batch[-1])
        train_loss = total_loss / len(train_loader.dataset)

        model.eval()
        with torch.no_grad():
            batch = [x.to(DEVICE) for x in next(iter(val_loader))]
            val_loss = criterion(forward_batch(model, batch), batch[-1]).item()
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        if val_loss < best_loss:
            best_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break
        if epoch % 50 == 0:
            print(f"Epoch {epoch:4d} | Train: {train_loss:.4f} | Val: {val_loss:.4f}")
    if best_state is None:
        raise RuntimeError("No valid validation loss was observed")
    model.load_state_dict(best_state)
    return best_loss, train_losses, val_losses, epoch


def predict_model(model, loader, forward_batch):
    model.eval()
    predictions, targets = [], []
    with torch.no_grad():
        for batch in loader:
            batch = [x.to(DEVICE) for x in batch]
            predictions.append(forward_batch(model, batch).detach().cpu().numpy().ravel())
            targets.append(batch[-1].detach().cpu().numpy().ravel())
    return np.concatenate(targets), np.concatenate(predictions)


def metrics(y_true, y_pred):
    residual = y_true - y_pred
    denominator = np.sum((y_true - y_true.mean()) ** 2)
    r2 = np.nan if denominator == 0 else 1.0 - np.sum(residual**2) / denominator
    return {
        "r2": float(r2),
        "rmse": float(np.sqrt(np.mean(residual**2))),
        "mae": float(np.mean(np.abs(residual))),
    }

def plot_parity(results, save_path, title="Parity plot"):
    """
    results: dict like {"Train": (y_true, y_pred), "Val": (...), "Test": (...)}
    All values must be in original LogS units.
    """
    colors = {"Train": "tab:blue", "Val": "tab:orange", "Test": "tab:green"}

    all_vals = np.concatenate([np.concatenate(v) for v in results.values()])
    lo, hi = all_vals.min(), all_vals.max()
    pad = 0.05 * (hi - lo)
    lims = (lo - pad, hi + pad)

    fig, axes = plt.subplots(1, len(results) + 1, figsize=(5 * (len(results) + 1), 5))

    # Combined panel
    ax = axes[0]
    for name, (yt, yp) in results.items():
        m = metrics(yt, yp)
        ax.scatter(yt, yp, s=40, alpha=0.5, color=colors.get(name), label=f"{name}")
    ax.plot(lims, lims, "k--", lw=1)
    ax.set(xlim=lims, ylim=lims)
    ax.set_xlabel("Experimental LogS", fontsize=18)
    ax.set_ylabel("Predicted LogS", fontsize=18)
    ax.set_aspect("equal")
    ax.legend()

    # One panel per split
    for ax, (name, (yt, yp)) in zip(axes[1:], results.items()):
        m = metrics(yt, yp)
        ax.scatter(yt, yp, s=12, alpha=0.5, color=colors.get(name))
        ax.plot(lims, lims, "k--", lw=1)
        ax.set(xlim=lims, ylim=lims, xlabel="Experimental LogS", ylabel="Predicted LogS",
               title=f"{name} (n={len(yt)})")
        ax.set_aspect("equal")
        ax.text(0.05, 0.95,
                f"R² = {m['r2']:.3f}\nRMSE = {m['rmse']:.3f}\nMAE = {m['mae']:.3f}",
                transform=ax.transAxes, va="top",
                bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))

    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)


In [11]:
# ============================================================================
# PCA - pretrain - Physics - Random row split
# ============================================================================
import argparse
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from rdkit import Chem
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA

def main(exp_data_path, pretrain_data_path, output_dir, variance):
    seed_everything(SEED)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    calculator = make_calculator()

    # ============================================================================
    # 1. STEP 0 — PRETRAIN ON COSMO/XTB DATASET WITH UNIQUE-SOLVENT PCA
    # ============================================================================
    print("\n--- STEP 0: PRETRAINING WITH UNIQUE-SOLVENT PCA (COSMO/XTB DATASET) ---")
    pretrain_df = pd.read_csv(pretrain_data_path)
    pretrain_df.columns = pretrain_df.columns.str.strip()
    pretrain_df = pretrain_df.dropna(
        subset=["SMILES_1", "Temperature", "comp_1", "LogS"]
    ).reset_index(drop=True)

    # Compute raw Mordred descriptors for pretrain dataset
    unique_smiles_pre = pretrain_df["SMILES_1"].unique()
    pre_lookup = {
        smi: smiles_to_mordred(smi, calculator) for smi in unique_smiles_pre
    }
    fp1_pre_raw = np.vstack([pre_lookup[s] for s in pretrain_df["SMILES_1"]])
    fp2_pre_raw = fp1_pre_raw.copy()  # Pure solvent placeholder

    x1_pretrain = (pretrain_df["comp_1"].values / 100.0).reshape(-1, 1)
    T_pretrain = pretrain_df["Temperature"].values.reshape(-1, 1)
    Y_pretrain = pretrain_df["LogS"].values.reshape(-1, 1)

    # Train/Val split for pretraining
    idx_all_pre = np.arange(len(fp1_pre_raw))
    idx_pre_train, idx_pre_val = train_test_split(
        idx_all_pre, test_size=0.2, random_state=SEED
    )

    # ── Unique-solvent PCA, placed RIGHT AFTER fp generation ───────────────
    # compact_descriptors_unique_solvents expects a df with both SMILES columns.
    # The pretrain frame only has SMILES_1 (pure solvent), so we mirror it into
    # SMILES_2 in a copy so the unique-SMILES extraction sees each chemical once.
    pretrain_df_for_pca = pretrain_df.copy()
    pretrain_df_for_pca["SMILES_2"] = pretrain_df_for_pca["SMILES_1"]

    fp1_pre, fp2_pre, standardizer, pca = compact_descriptors_unique_solvents(
        pretrain_df_for_pca, fp1_pre_raw, fp2_pre_raw,
        idx_pre_train, variance=variance
    )
    # ───────────────────────────────────────────────────────────────────────

    print(f"Pretrain dataset size: {len(pretrain_df)}")
    print(f"PCA components: {pca.n_components_} | "
          f"Explained variance: {pca.explained_variance_ratio_.sum():.6f}")

    # Create PyTorch DataLoaders for pretraining
    split_pre = lambda a: (a[idx_pre_train], a[idx_pre_val])
    fp1_pre_tr, fp1_pre_va = split_pre(fp1_pre)
    fp2_pre_tr, fp2_pre_va = split_pre(fp2_pre)
    x_pre_tr, x_pre_va = split_pre(x1_pretrain)
    t_pre_tr, t_pre_va = split_pre(T_pretrain)
    y_pre_tr, y_pre_va = split_pre(Y_pretrain)

    train_loader_pre, val_loader_pre, _ = make_loaders(
        [fp1_pre_tr, fp2_pre_tr, x_pre_tr, t_pre_tr, y_pre_tr],
        [fp1_pre_va, fp2_pre_va, x_pre_va, t_pre_va, y_pre_va],
        [fp1_pre_va, fp2_pre_va, x_pre_va, t_pre_va, y_pre_va],
    )

    # Initialize and train pretraining model
    pretrain_model = WeightedSumFingerprintModel(n_desc=fp1_pre.shape[1]).to(DEVICE)
    print(f"Trainable parameters: {pretrain_model.count_parameters()}")

    forward = lambda m, batch: m(batch[0], batch[1], batch[2], batch[3])
    best_val_pre, _, _, epochs_pre = train_model(
        pretrain_model, train_loader_pre, val_loader_pre, forward
    )
    print(f"[Pretrain Complete] Best val loss: {best_val_pre:.6f} (Epoch {epochs_pre})")

    # Save pretrain model checkpoint and PCA transformer
    torch.save({
        "model_state_dict": pretrain_model.state_dict(),
        "model_config": {"n_desc": fp1_pre.shape[1]},
        "seed": SEED,
    }, output_dir / "pretrained_pca_unique_solvents_cosmo_model.pt")
    with (output_dir / "pretrained_pca_unique_solvents_cosmo_preprocessor.pkl").open("wb") as handle:
        pickle.dump({
            "fp_standardizer": standardizer,
            "pca": pca,
            "pca_variance_target": variance,
        }, handle)

    # ============================================================================
    # 2. STEP 1 — FINE-TUNE ON EXPERIMENTAL DATA (REUSE PRETRAINED PCA SPACE)
    # ============================================================================
    print("\n--- STEP 1: FINE-TUNING (EXPERIMENTAL DATASET) ---")
    df, x1, temperature, target = load_raw_data(exp_data_path)
    idx_train, idx_val, idx_test, sig, keys  = random_row_split(df, seed=SEED)

    print(f"Experimental dataset size: {len(df)}")
    print(f"Split sizes: Train={len(idx_train)} | Val={len(idx_val)} | "
          f"Test={len(idx_test)}")

    fp1_exp_raw, fp2_exp_raw, descriptor_lookup = compute_fingerprints(df, calculator)

    # Transform experimental features using the PRETRAINED standardizer + PCA space
    fp1_exp_std = standardizer.transform(fp1_exp_raw)
    fp2_exp_std = standardizer.transform(fp2_exp_raw)
    fp1_exp = pca.transform(fp1_exp_std)
    fp2_exp = pca.transform(fp2_exp_std)

    split_exp = lambda a: (a[idx_train], a[idx_val], a[idx_test])
    fp1_tr, fp1_va, fp1_te = split_exp(fp1_exp)
    fp2_tr, fp2_va, fp2_te = split_exp(fp2_exp)
    x_tr, x_va, x_te = split_exp(x1)
    t_tr, t_va, t_te = split_exp(temperature)
    y_tr, y_va, y_te = split_exp(target)

    train_loader, val_loader, test_loader = make_loaders(
        [fp1_tr, fp2_tr, x_tr, t_tr, y_tr],
        [fp1_va, fp2_va, x_va, t_va, y_va],
        [fp1_te, fp2_te, x_te, t_te, y_te],
    )

    # Load pretrained weights into fine-tuning model
    finetune_model = WeightedSumFingerprintModel(n_desc=fp1_exp.shape[1]).to(DEVICE)
    checkpoint = torch.load(output_dir / "pretrained_pca_unique_solvents_cosmo_model.pt")
    finetune_model.load_state_dict(checkpoint["model_state_dict"])
    print("Pretrained weights loaded successfully into fine-tuning architecture.")

    best_val_ft, _, _, epochs_ft = train_model(
        finetune_model, train_loader, val_loader, forward
    )
    print(f"[Fine-tune Complete] Best val loss: {best_val_ft:.6f} (Epoch {epochs_ft})")

    # Evaluate metrics across all splits
    y_true_train, y_pred_train = predict_model(finetune_model, train_loader, forward)
    y_true_val, y_pred_val = predict_model(finetune_model, val_loader, forward)
    y_true_test, y_pred_test = predict_model(finetune_model, test_loader, forward)

    m_train = metrics(y_true_train, y_pred_train)
    m_val = metrics(y_true_val, y_pred_val)
    m_test = metrics(y_true_test, y_pred_test)

    print("\n--- FINE-TUNED METRICS (UNIQUE-SOLVENT PCA) ---")
    print(f"Train | R²={m_train['r2']:.4f} RMSE={m_train['rmse']:.4f} MAE={m_train['mae']:.4f}")
    print(f"Val   | R²={m_val['r2']:.4f} RMSE={m_val['rmse']:.4f} MAE={m_val['mae']:.4f}")
    print(f"Test  | R²={m_test['r2']:.4f} RMSE={m_test['rmse']:.4f} MAE={m_test['mae']:.4f}")

    plot_parity(
        {"Train": (y_true_train, y_pred_train), "Val": (y_true_val, y_pred_val), "Test": (y_true_test, y_pred_test)},
        output_dir / "Final_Model_Random_Row_Split.png",
        title="Clozapine LogS Prediction Parity Plot"
    )


parser = argparse.ArgumentParser()
parser.add_argument("--exp-data", default="Clozapine_dataset.csv")
parser.add_argument("--pretrain-data", default="combined_file_XTB.csv")
parser.add_argument("--output-dir", default="Results")
parser.add_argument("--variance", type=float, default=0.95)
args = parser.parse_args([])

main(args.exp_data, args.pretrain_data, args.output_dir, args.variance)


--- STEP 0: PRETRAINING WITH UNIQUE-SOLVENT PCA (COSMO/XTB DATASET) ---
[TrainOnlyStandardizer] Descriptors: 681 total | 191 dropped (NaN) | 178 dropped (constant) | 312 remaining
Pretrain dataset size: 6000
PCA components: 61 | Explained variance: 0.950345
Trainable parameters: 6882
Epoch   50 | Train: 0.0285 | Val: 0.0153
Epoch  100 | Train: 0.0199 | Val: 0.0106
Epoch  150 | Train: 0.0175 | Val: 0.0096
Epoch  200 | Train: 0.0172 | Val: 0.0105
Epoch  250 | Train: 0.0174 | Val: 0.0106
[Pretrain Complete] Best val loss: 0.009034 (Epoch 279)

--- STEP 1: FINE-TUNING (EXPERIMENTAL DATASET) ---
Experimental dataset size: 684
Split sizes: Train=478 | Val=103 | Test=103
Pretrained weights loaded successfully into fine-tuning architecture.
Epoch   50 | Train: 0.0857 | Val: 0.0708
Epoch  100 | Train: 0.0830 | Val: 0.0594
Epoch  150 | Train: 0.0653 | Val: 0.0568
Epoch  200 | Train: 0.0640 | Val: 0.0573
Epoch  250 | Train: 0.0542 | Val: 0.0656
Epoch  300 | Train: 0.0403 | Val: 0.0183
[Fine-tune

In [12]:
# ============================================================================
# PCA - pretrain - Physics - Water + 2-Propanol
# ============================================================================
import argparse
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from rdkit import Chem
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA

# Target components for the explicit test set
SMILES__1 = "O"
SMILES__2 = "CC(C)O"

def main(exp_data_path, pretrain_data_path, output_dir, variance):
    seed_everything(SEED)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    calculator = make_calculator()

    # ============================================================================
    # 1. STEP 0 — PRETRAIN ON COSMO/XTB DATASET WITH UNIQUE-SOLVENT PCA
    # ============================================================================
    print("\n--- STEP 0: PRETRAINING WITH UNIQUE-SOLVENT PCA (COSMO/XTB DATASET) ---")
    pretrain_df = pd.read_csv(pretrain_data_path)
    pretrain_df.columns = pretrain_df.columns.str.strip()
    pretrain_df = pretrain_df.dropna(
        subset=["SMILES_1", "Temperature", "comp_1", "LogS"]
    ).reset_index(drop=True)

    # Compute raw Mordred descriptors for pretrain dataset
    unique_smiles_pre = pretrain_df["SMILES_1"].unique()
    pre_lookup = {
        smi: smiles_to_mordred(smi, calculator) for smi in unique_smiles_pre
    }
    fp1_pre_raw = np.vstack([pre_lookup[s] for s in pretrain_df["SMILES_1"]])
    fp2_pre_raw = fp1_pre_raw.copy()  # Pure solvent placeholder

    x1_pretrain = (pretrain_df["comp_1"].values / 100.0).reshape(-1, 1)
    T_pretrain = pretrain_df["Temperature"].values.reshape(-1, 1)
    Y_pretrain = pretrain_df["LogS"].values.reshape(-1, 1)

    # Train/Val split for pretraining
    idx_all_pre = np.arange(len(fp1_pre_raw))
    idx_pre_train, idx_pre_val = train_test_split(
        idx_all_pre, test_size=0.2, random_state=SEED
    )

    # ── Unique-solvent PCA, placed RIGHT AFTER fp generation ───────────────
    # compact_descriptors_unique_solvents expects a df with both SMILES columns.
    # The pretrain frame only has SMILES_1 (pure solvent), so we mirror it into
    # SMILES_2 in a copy so the unique-SMILES extraction sees each chemical once.
    pretrain_df_for_pca = pretrain_df.copy()
    pretrain_df_for_pca["SMILES_2"] = pretrain_df_for_pca["SMILES_1"]

    fp1_pre, fp2_pre, standardizer, pca = compact_descriptors_unique_solvents(
        pretrain_df_for_pca, fp1_pre_raw, fp2_pre_raw,
        idx_pre_train, variance=variance
    )
    # ───────────────────────────────────────────────────────────────────────

    print(f"Pretrain dataset size: {len(pretrain_df)}")
    print(f"PCA components: {pca.n_components_} | "
          f"Explained variance: {pca.explained_variance_ratio_.sum():.6f}")

    # Create PyTorch DataLoaders for pretraining
    split_pre = lambda a: (a[idx_pre_train], a[idx_pre_val])
    fp1_pre_tr, fp1_pre_va = split_pre(fp1_pre)
    fp2_pre_tr, fp2_pre_va = split_pre(fp2_pre)
    x_pre_tr, x_pre_va = split_pre(x1_pretrain)
    t_pre_tr, t_pre_va = split_pre(T_pretrain)
    y_pre_tr, y_pre_va = split_pre(Y_pretrain)

    train_loader_pre, val_loader_pre, _ = make_loaders(
        [fp1_pre_tr, fp2_pre_tr, x_pre_tr, t_pre_tr, y_pre_tr],
        [fp1_pre_va, fp2_pre_va, x_pre_va, t_pre_va, y_pre_va],
        [fp1_pre_va, fp2_pre_va, x_pre_va, t_pre_va, y_pre_va],
    )

    # Initialize and train pretraining model
    pretrain_model = WeightedSumFingerprintModel(n_desc=fp1_pre.shape[1]).to(DEVICE)
    print(f"Trainable parameters: {pretrain_model.count_parameters()}")

    forward = lambda m, batch: m(batch[0], batch[1], batch[2], batch[3])
    best_val_pre, _, _, epochs_pre = train_model(
        pretrain_model, train_loader_pre, val_loader_pre, forward
    )
    print(f"[Pretrain Complete] Best val loss: {best_val_pre:.6f} (Epoch {epochs_pre})")

    # Save pretrain model checkpoint and PCA transformer
    torch.save({
        "model_state_dict": pretrain_model.state_dict(),
        "model_config": {"n_desc": fp1_pre.shape[1]},
        "seed": SEED,
    }, output_dir / "pretrained_pca_unique_solvents_cosmo_model.pt")
    with (output_dir / "pretrained_pca_unique_solvents_cosmo_preprocessor.pkl").open("wb") as handle:
        pickle.dump({
            "fp_standardizer": standardizer,
            "pca": pca,
            "pca_variance_target": variance,
        }, handle)

    # ============================================================================
    # 2. STEP 1 — FINE-TUNE ON EXPERIMENTAL DATA (REUSE PRETRAINED PCA SPACE)
    # ============================================================================
    print("\n--- STEP 1: FINE-TUNING (EXPERIMENTAL DATASET) ---")
    df, x1, temperature, target = load_raw_data(exp_data_path)
    idx_train, idx_val, idx_test = group_split(df, SMILES__1, SMILES__2, seed=SEED)

    print(f"Experimental dataset size: {len(df)}")
    print(f"Split sizes: Train={len(idx_train)} | Val={len(idx_val)} | "
          f"Test ={len(idx_test)}")

    fp1_exp_raw, fp2_exp_raw, descriptor_lookup = compute_fingerprints(df, calculator)

    # Transform experimental features using the PRETRAINED standardizer + PCA space
    fp1_exp_std = standardizer.transform(fp1_exp_raw)
    fp2_exp_std = standardizer.transform(fp2_exp_raw)
    fp1_exp = pca.transform(fp1_exp_std)
    fp2_exp = pca.transform(fp2_exp_std)

    split_exp = lambda a: (a[idx_train], a[idx_val], a[idx_test])
    fp1_tr, fp1_va, fp1_te = split_exp(fp1_exp)
    fp2_tr, fp2_va, fp2_te = split_exp(fp2_exp)
    x_tr, x_va, x_te = split_exp(x1)
    t_tr, t_va, t_te = split_exp(temperature)
    y_tr, y_va, y_te = split_exp(target)

    train_loader, val_loader, test_loader = make_loaders(
        [fp1_tr, fp2_tr, x_tr, t_tr, y_tr],
        [fp1_va, fp2_va, x_va, t_va, y_va],
        [fp1_te, fp2_te, x_te, t_te, y_te],
    )

    # Load pretrained weights into fine-tuning model
    finetune_model = WeightedSumFingerprintModel(n_desc=fp1_exp.shape[1]).to(DEVICE)
    checkpoint = torch.load(output_dir / "pretrained_pca_unique_solvents_cosmo_model.pt")
    finetune_model.load_state_dict(checkpoint["model_state_dict"])
    print("Pretrained weights loaded successfully into fine-tuning architecture.")

    best_val_ft, _, _, epochs_ft = train_model(
        finetune_model, train_loader, val_loader, forward
    )
    print(f"[Fine-tune Complete] Best val loss: {best_val_ft:.6f} (Epoch {epochs_ft})")

    # Evaluate metrics across all splits
    y_true_train, y_pred_train = predict_model(finetune_model, train_loader, forward)
    y_true_val, y_pred_val = predict_model(finetune_model, val_loader, forward)
    y_true_test, y_pred_test = predict_model(finetune_model, test_loader, forward)

    m_train = metrics(y_true_train, y_pred_train)
    m_val = metrics(y_true_val, y_pred_val)
    m_test = metrics(y_true_test, y_pred_test)

    print("\n--- FINE-TUNED METRICS (UNIQUE-SOLVENT PCA) ---")
    print(f"Train | R²={m_train['r2']:.4f} RMSE={m_train['rmse']:.4f} MAE={m_train['mae']:.4f}")
    print(f"Val   | R²={m_val['r2']:.4f} RMSE={m_val['rmse']:.4f} MAE={m_val['mae']:.4f}")
    print(f"Test  | R²={m_test['r2']:.4f} RMSE={m_test['rmse']:.4f} MAE={m_test['mae']:.4f}")

    plot_parity(
        {"Train": (y_true_train, y_pred_train), "Val": (y_true_val, y_pred_val), "Test": (y_true_test, y_pred_test)},
        output_dir / "Final_Model_Water_2-Propanol.png",
        title="Clozapine LogS Prediction Parity Plot"
    )


parser = argparse.ArgumentParser()
parser.add_argument("--exp-data", default="Clozapine_dataset.csv")
parser.add_argument("--pretrain-data", default="combined_file_XTB.csv")
parser.add_argument("--output-dir", default="Results")
parser.add_argument("--variance", type=float, default=0.95)
args = parser.parse_args([])

main(args.exp_data, args.pretrain_data, args.output_dir, args.variance)


--- STEP 0: PRETRAINING WITH UNIQUE-SOLVENT PCA (COSMO/XTB DATASET) ---
[TrainOnlyStandardizer] Descriptors: 681 total | 191 dropped (NaN) | 178 dropped (constant) | 312 remaining
Pretrain dataset size: 6000
PCA components: 61 | Explained variance: 0.950345
Trainable parameters: 6882
Epoch   50 | Train: 0.0285 | Val: 0.0153
Epoch  100 | Train: 0.0199 | Val: 0.0106
Epoch  150 | Train: 0.0175 | Val: 0.0096
Epoch  200 | Train: 0.0172 | Val: 0.0105
Epoch  250 | Train: 0.0174 | Val: 0.0106
[Pretrain Complete] Best val loss: 0.009034 (Epoch 279)

--- STEP 1: FINE-TUNING (EXPERIMENTAL DATASET) ---
Experimental dataset size: 684
Split sizes: Train=495 | Val=90 | Test =99
Pretrained weights loaded successfully into fine-tuning architecture.
Epoch   50 | Train: 0.1005 | Val: 0.0271
Epoch  100 | Train: 0.0909 | Val: 0.0408
Epoch  150 | Train: 0.0822 | Val: 0.0276
Epoch  200 | Train: 0.0764 | Val: 0.0105
[Fine-tune Complete] Best val loss: 0.008769 (Epoch 222)

--- FINE-TUNED METRICS (UNIQUE-SOLV

In [3]:
# ============================================================================
# PCA - pretrain - Physics - Water + NMP
# ============================================================================
import argparse
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from rdkit import Chem
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA

# Target components for the explicit test set
SMILES__1 = "O"
SMILES__2 = "CN1CCCC1=O"   # N-Methyl-2-pyrrolidone

def main(exp_data_path, pretrain_data_path, output_dir, variance):
    seed_everything(SEED)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    calculator = make_calculator()

    # ============================================================================
    # 1. STEP 0 — PRETRAIN ON COSMO/XTB DATASET WITH UNIQUE-SOLVENT PCA
    # ============================================================================
    print("\n--- STEP 0: PRETRAINING WITH UNIQUE-SOLVENT PCA (COSMO/XTB DATASET) ---")
    pretrain_df = pd.read_csv(pretrain_data_path)
    pretrain_df.columns = pretrain_df.columns.str.strip()
    pretrain_df = pretrain_df.dropna(
        subset=["SMILES_1", "Temperature", "comp_1", "LogS"]
    ).reset_index(drop=True)

    # Compute raw Mordred descriptors for pretrain dataset
    unique_smiles_pre = pretrain_df["SMILES_1"].unique()
    pre_lookup = {
        smi: smiles_to_mordred(smi, calculator) for smi in unique_smiles_pre
    }
    fp1_pre_raw = np.vstack([pre_lookup[s] for s in pretrain_df["SMILES_1"]])
    fp2_pre_raw = fp1_pre_raw.copy()  # Pure solvent placeholder

    x1_pretrain = (pretrain_df["comp_1"].values / 100.0).reshape(-1, 1)
    T_pretrain = pretrain_df["Temperature"].values.reshape(-1, 1)
    Y_pretrain = pretrain_df["LogS"].values.reshape(-1, 1)

    # Train/Val split for pretraining
    idx_all_pre = np.arange(len(fp1_pre_raw))
    idx_pre_train, idx_pre_val = train_test_split(
        idx_all_pre, test_size=0.2, random_state=SEED
    )

    # ── Unique-solvent PCA, placed RIGHT AFTER fp generation ───────────────
    # compact_descriptors_unique_solvents expects a df with both SMILES columns.
    # The pretrain frame only has SMILES_1 (pure solvent), so we mirror it into
    # SMILES_2 in a copy so the unique-SMILES extraction sees each chemical once.
    pretrain_df_for_pca = pretrain_df.copy()
    pretrain_df_for_pca["SMILES_2"] = pretrain_df_for_pca["SMILES_1"]

    fp1_pre, fp2_pre, standardizer, pca = compact_descriptors_unique_solvents(
        pretrain_df_for_pca, fp1_pre_raw, fp2_pre_raw,
        idx_pre_train, variance=variance
    )
    # ───────────────────────────────────────────────────────────────────────

    print(f"Pretrain dataset size: {len(pretrain_df)}")
    print(f"PCA components: {pca.n_components_} | "
          f"Explained variance: {pca.explained_variance_ratio_.sum():.6f}")

    # Create PyTorch DataLoaders for pretraining
    split_pre = lambda a: (a[idx_pre_train], a[idx_pre_val])
    fp1_pre_tr, fp1_pre_va = split_pre(fp1_pre)
    fp2_pre_tr, fp2_pre_va = split_pre(fp2_pre)
    x_pre_tr, x_pre_va = split_pre(x1_pretrain)
    t_pre_tr, t_pre_va = split_pre(T_pretrain)
    y_pre_tr, y_pre_va = split_pre(Y_pretrain)

    train_loader_pre, val_loader_pre, _ = make_loaders(
        [fp1_pre_tr, fp2_pre_tr, x_pre_tr, t_pre_tr, y_pre_tr],
        [fp1_pre_va, fp2_pre_va, x_pre_va, t_pre_va, y_pre_va],
        [fp1_pre_va, fp2_pre_va, x_pre_va, t_pre_va, y_pre_va],
    )

    # Initialize and train pretraining model
    pretrain_model = WeightedSumFingerprintModel(n_desc=fp1_pre.shape[1]).to(DEVICE)
    print(f"Trainable parameters: {pretrain_model.count_parameters()}")

    forward = lambda m, batch: m(batch[0], batch[1], batch[2], batch[3])
    best_val_pre, _, _, epochs_pre = train_model(
        pretrain_model, train_loader_pre, val_loader_pre, forward
    )
    print(f"[Pretrain Complete] Best val loss: {best_val_pre:.6f} (Epoch {epochs_pre})")

    # Save pretrain model checkpoint and PCA transformer
    torch.save({
        "model_state_dict": pretrain_model.state_dict(),
        "model_config": {"n_desc": fp1_pre.shape[1]},
        "seed": SEED,
    }, output_dir / "pretrained_pca_unique_solvents_cosmo_model.pt")
    with (output_dir / "pretrained_pca_unique_solvents_cosmo_preprocessor.pkl").open("wb") as handle:
        pickle.dump({
            "fp_standardizer": standardizer,
            "pca": pca,
            "pca_variance_target": variance,
        }, handle)

    # ============================================================================
    # 2. STEP 1 — FINE-TUNE ON EXPERIMENTAL DATA (REUSE PRETRAINED PCA SPACE)
    # ============================================================================
    print("\n--- STEP 1: FINE-TUNING (EXPERIMENTAL DATASET) ---")
    df, x1, temperature, target = load_raw_data(exp_data_path)
    idx_train, idx_val, idx_test = group_split(df, SMILES__1, SMILES__2, seed=SEED)

    print(f"Experimental dataset size: {len(df)}")
    print(f"Split sizes: Train={len(idx_train)} | Val={len(idx_val)} | "
          f"Test ={len(idx_test)}")

    fp1_exp_raw, fp2_exp_raw, descriptor_lookup = compute_fingerprints(df, calculator)

    # Transform experimental features using the PRETRAINED standardizer + PCA space
    fp1_exp_std = standardizer.transform(fp1_exp_raw)
    fp2_exp_std = standardizer.transform(fp2_exp_raw)
    fp1_exp = pca.transform(fp1_exp_std)
    fp2_exp = pca.transform(fp2_exp_std)

    split_exp = lambda a: (a[idx_train], a[idx_val], a[idx_test])
    fp1_tr, fp1_va, fp1_te = split_exp(fp1_exp)
    fp2_tr, fp2_va, fp2_te = split_exp(fp2_exp)
    x_tr, x_va, x_te = split_exp(x1)
    t_tr, t_va, t_te = split_exp(temperature)
    y_tr, y_va, y_te = split_exp(target)

    train_loader, val_loader, test_loader = make_loaders(
        [fp1_tr, fp2_tr, x_tr, t_tr, y_tr],
        [fp1_va, fp2_va, x_va, t_va, y_va],
        [fp1_te, fp2_te, x_te, t_te, y_te],
    )

    # Load pretrained weights into fine-tuning model
    finetune_model = WeightedSumFingerprintModel(n_desc=fp1_exp.shape[1]).to(DEVICE)
    checkpoint = torch.load(output_dir / "pretrained_pca_unique_solvents_cosmo_model.pt")
    finetune_model.load_state_dict(checkpoint["model_state_dict"])
    print("Pretrained weights loaded successfully into fine-tuning architecture.")

    best_val_ft, _, _, epochs_ft = train_model(
        finetune_model, train_loader, val_loader, forward
    )
    print(f"[Fine-tune Complete] Best val loss: {best_val_ft:.6f} (Epoch {epochs_ft})")

    # Evaluate metrics across all splits
    y_true_train, y_pred_train = predict_model(finetune_model, train_loader, forward)
    y_true_val, y_pred_val = predict_model(finetune_model, val_loader, forward)
    y_true_test, y_pred_test = predict_model(finetune_model, test_loader, forward)

    m_train = metrics(y_true_train, y_pred_train)
    m_val = metrics(y_true_val, y_pred_val)
    m_test = metrics(y_true_test, y_pred_test)

    print("\n--- FINE-TUNED METRICS (UNIQUE-SOLVENT PCA) ---")
    print(f"Train | R²={m_train['r2']:.4f} RMSE={m_train['rmse']:.4f} MAE={m_train['mae']:.4f}")
    print(f"Val   | R²={m_val['r2']:.4f} RMSE={m_val['rmse']:.4f} MAE={m_val['mae']:.4f}")
    print(f"Test  | R²={m_test['r2']:.4f} RMSE={m_test['rmse']:.4f} MAE={m_test['mae']:.4f}")

    plot_parity(
        {"Train": (y_true_train, y_pred_train), "Val": (y_true_val, y_pred_val), "Test": (y_true_test, y_pred_test)},
        output_dir / "Final_Model_Water_NMP.png",
        title="Clozapine LogS Prediction Parity Plot"
    )


parser = argparse.ArgumentParser()
parser.add_argument("--exp-data", default="Clozapine_dataset.csv")
parser.add_argument("--pretrain-data", default="combined_file_XTB.csv")
parser.add_argument("--output-dir", default="Results")
parser.add_argument("--variance", type=float, default=0.95)
args = parser.parse_args([])

main(args.exp_data, args.pretrain_data, args.output_dir, args.variance)


--- STEP 0: PRETRAINING WITH UNIQUE-SOLVENT PCA (COSMO/XTB DATASET) ---
[TrainOnlyStandardizer] Descriptors: 681 total | 191 dropped (NaN) | 178 dropped (constant) | 312 remaining
Pretrain dataset size: 6000
PCA components: 61 | Explained variance: 0.950345
Trainable parameters: 6882
Epoch   50 | Train: 0.0285 | Val: 0.0153
Epoch  100 | Train: 0.0199 | Val: 0.0106
Epoch  150 | Train: 0.0175 | Val: 0.0096
Epoch  200 | Train: 0.0172 | Val: 0.0105
Epoch  250 | Train: 0.0174 | Val: 0.0106
[Pretrain Complete] Best val loss: 0.009034 (Epoch 279)

--- STEP 1: FINE-TUNING (EXPERIMENTAL DATASET) ---
Experimental dataset size: 684
Split sizes: Train=497 | Val=88 | Test =99
Pretrained weights loaded successfully into fine-tuning architecture.
Epoch   50 | Train: 0.0919 | Val: 0.0220
Epoch  100 | Train: 0.0836 | Val: 0.0172
[Fine-tune Complete] Best val loss: 0.008554 (Epoch 130)

--- FINE-TUNED METRICS (UNIQUE-SOLVENT PCA) ---
Train | R²=0.9318 RMSE=0.2579 MAE=0.1530
Val   | R²=0.9558 RMSE=0.092

In [4]:
# ============================================================================
# PCA - pretrain - Physics - Acetone + Toluene
# ============================================================================
import argparse
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from rdkit import Chem
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA

# Target components for the explicit test set
SMILES__1 = "Cc1ccccc1"
SMILES__2 = "CC(=O)C"

def main(exp_data_path, pretrain_data_path, output_dir, variance):
    seed_everything(SEED)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    calculator = make_calculator()

    # ============================================================================
    # 1. STEP 0 — PRETRAIN ON COSMO/XTB DATASET WITH UNIQUE-SOLVENT PCA
    # ============================================================================
    print("\n--- STEP 0: PRETRAINING WITH UNIQUE-SOLVENT PCA (COSMO/XTB DATASET) ---")
    pretrain_df = pd.read_csv(pretrain_data_path)
    pretrain_df.columns = pretrain_df.columns.str.strip()
    pretrain_df = pretrain_df.dropna(
        subset=["SMILES_1", "Temperature", "comp_1", "LogS"]
    ).reset_index(drop=True)

    # Compute raw Mordred descriptors for pretrain dataset
    unique_smiles_pre = pretrain_df["SMILES_1"].unique()
    pre_lookup = {
        smi: smiles_to_mordred(smi, calculator) for smi in unique_smiles_pre
    }
    fp1_pre_raw = np.vstack([pre_lookup[s] for s in pretrain_df["SMILES_1"]])
    fp2_pre_raw = fp1_pre_raw.copy()  # Pure solvent placeholder

    x1_pretrain = (pretrain_df["comp_1"].values / 100.0).reshape(-1, 1)
    T_pretrain = pretrain_df["Temperature"].values.reshape(-1, 1)
    Y_pretrain = pretrain_df["LogS"].values.reshape(-1, 1)

    # Train/Val split for pretraining
    idx_all_pre = np.arange(len(fp1_pre_raw))
    idx_pre_train, idx_pre_val = train_test_split(
        idx_all_pre, test_size=0.2, random_state=SEED
    )

    # ── Unique-solvent PCA, placed RIGHT AFTER fp generation ───────────────
    # compact_descriptors_unique_solvents expects a df with both SMILES columns.
    # The pretrain frame only has SMILES_1 (pure solvent), so we mirror it into
    # SMILES_2 in a copy so the unique-SMILES extraction sees each chemical once.
    pretrain_df_for_pca = pretrain_df.copy()
    pretrain_df_for_pca["SMILES_2"] = pretrain_df_for_pca["SMILES_1"]

    fp1_pre, fp2_pre, standardizer, pca = compact_descriptors_unique_solvents(
        pretrain_df_for_pca, fp1_pre_raw, fp2_pre_raw,
        idx_pre_train, variance=variance
    )
    # ───────────────────────────────────────────────────────────────────────

    print(f"Pretrain dataset size: {len(pretrain_df)}")
    print(f"PCA components: {pca.n_components_} | "
          f"Explained variance: {pca.explained_variance_ratio_.sum():.6f}")

    # Create PyTorch DataLoaders for pretraining
    split_pre = lambda a: (a[idx_pre_train], a[idx_pre_val])
    fp1_pre_tr, fp1_pre_va = split_pre(fp1_pre)
    fp2_pre_tr, fp2_pre_va = split_pre(fp2_pre)
    x_pre_tr, x_pre_va = split_pre(x1_pretrain)
    t_pre_tr, t_pre_va = split_pre(T_pretrain)
    y_pre_tr, y_pre_va = split_pre(Y_pretrain)

    train_loader_pre, val_loader_pre, _ = make_loaders(
        [fp1_pre_tr, fp2_pre_tr, x_pre_tr, t_pre_tr, y_pre_tr],
        [fp1_pre_va, fp2_pre_va, x_pre_va, t_pre_va, y_pre_va],
        [fp1_pre_va, fp2_pre_va, x_pre_va, t_pre_va, y_pre_va],
    )

    # Initialize and train pretraining model
    pretrain_model = WeightedSumFingerprintModel(n_desc=fp1_pre.shape[1]).to(DEVICE)
    print(f"Trainable parameters: {pretrain_model.count_parameters()}")

    forward = lambda m, batch: m(batch[0], batch[1], batch[2], batch[3])
    best_val_pre, _, _, epochs_pre = train_model(
        pretrain_model, train_loader_pre, val_loader_pre, forward
    )
    print(f"[Pretrain Complete] Best val loss: {best_val_pre:.6f} (Epoch {epochs_pre})")

    # Save pretrain model checkpoint and PCA transformer
    torch.save({
        "model_state_dict": pretrain_model.state_dict(),
        "model_config": {"n_desc": fp1_pre.shape[1]},
        "seed": SEED,
    }, output_dir / "pretrained_pca_unique_solvents_cosmo_model.pt")
    with (output_dir / "pretrained_pca_unique_solvents_cosmo_preprocessor.pkl").open("wb") as handle:
        pickle.dump({
            "fp_standardizer": standardizer,
            "pca": pca,
            "pca_variance_target": variance,
        }, handle)

    # ============================================================================
    # 2. STEP 1 — FINE-TUNE ON EXPERIMENTAL DATA (REUSE PRETRAINED PCA SPACE)
    # ============================================================================
    print("\n--- STEP 1: FINE-TUNING (EXPERIMENTAL DATASET) ---")
    df, x1, temperature, target = load_raw_data(exp_data_path)
    idx_train, idx_val, idx_test = group_split(df, SMILES__1, SMILES__2, seed=SEED)

    print(f"Experimental dataset size: {len(df)}")
    print(f"Split sizes: Train={len(idx_train)} | Val={len(idx_val)} | "
          f"Test ={len(idx_test)}")

    fp1_exp_raw, fp2_exp_raw, descriptor_lookup = compute_fingerprints(df, calculator)

    # Transform experimental features using the PRETRAINED standardizer + PCA space
    fp1_exp_std = standardizer.transform(fp1_exp_raw)
    fp2_exp_std = standardizer.transform(fp2_exp_raw)
    fp1_exp = pca.transform(fp1_exp_std)
    fp2_exp = pca.transform(fp2_exp_std)

    split_exp = lambda a: (a[idx_train], a[idx_val], a[idx_test])
    fp1_tr, fp1_va, fp1_te = split_exp(fp1_exp)
    fp2_tr, fp2_va, fp2_te = split_exp(fp2_exp)
    x_tr, x_va, x_te = split_exp(x1)
    t_tr, t_va, t_te = split_exp(temperature)
    y_tr, y_va, y_te = split_exp(target)

    train_loader, val_loader, test_loader = make_loaders(
        [fp1_tr, fp2_tr, x_tr, t_tr, y_tr],
        [fp1_va, fp2_va, x_va, t_va, y_va],
        [fp1_te, fp2_te, x_te, t_te, y_te],
    )

    # Load pretrained weights into fine-tuning model
    finetune_model = WeightedSumFingerprintModel(n_desc=fp1_exp.shape[1]).to(DEVICE)
    checkpoint = torch.load(output_dir / "pretrained_pca_unique_solvents_cosmo_model.pt")
    finetune_model.load_state_dict(checkpoint["model_state_dict"])
    print("Pretrained weights loaded successfully into fine-tuning architecture.")

    best_val_ft, _, _, epochs_ft = train_model(
        finetune_model, train_loader, val_loader, forward
    )
    print(f"[Fine-tune Complete] Best val loss: {best_val_ft:.6f} (Epoch {epochs_ft})")

    # Evaluate metrics across all splits
    y_true_train, y_pred_train = predict_model(finetune_model, train_loader, forward)
    y_true_val, y_pred_val = predict_model(finetune_model, val_loader, forward)
    y_true_test, y_pred_test = predict_model(finetune_model, test_loader, forward)

    m_train = metrics(y_true_train, y_pred_train)
    m_val = metrics(y_true_val, y_pred_val)
    m_test = metrics(y_true_test, y_pred_test)

    print("\n--- FINE-TUNED METRICS (UNIQUE-SOLVENT PCA) ---")
    print(f"Train | R²={m_train['r2']:.4f} RMSE={m_train['rmse']:.4f} MAE={m_train['mae']:.4f}")
    print(f"Val   | R²={m_val['r2']:.4f} RMSE={m_val['rmse']:.4f} MAE={m_val['mae']:.4f}")
    print(f"Test  | R²={m_test['r2']:.4f} RMSE={m_test['rmse']:.4f} MAE={m_test['mae']:.4f}")

    plot_parity(
        {"Train": (y_true_train, y_pred_train), "Val": (y_true_val, y_pred_val), "Test": (y_true_test, y_pred_test)},
        output_dir / "Final_Model_Acetone_Toluene.png",
        title="Clozapine LogS Prediction Parity Plot"
    )


parser = argparse.ArgumentParser()
parser.add_argument("--exp-data", default="Clozapine_dataset.csv")
parser.add_argument("--pretrain-data", default="combined_file_XTB.csv")
parser.add_argument("--output-dir", default="Results")
parser.add_argument("--variance", type=float, default=0.95)
args = parser.parse_args([])

main(args.exp_data, args.pretrain_data, args.output_dir, args.variance)


--- STEP 0: PRETRAINING WITH UNIQUE-SOLVENT PCA (COSMO/XTB DATASET) ---
[TrainOnlyStandardizer] Descriptors: 681 total | 191 dropped (NaN) | 178 dropped (constant) | 312 remaining
Pretrain dataset size: 6000
PCA components: 61 | Explained variance: 0.950345
Trainable parameters: 6882
Epoch   50 | Train: 0.0285 | Val: 0.0153
Epoch  100 | Train: 0.0199 | Val: 0.0106
Epoch  150 | Train: 0.0175 | Val: 0.0096
Epoch  200 | Train: 0.0172 | Val: 0.0105
Epoch  250 | Train: 0.0174 | Val: 0.0106
[Pretrain Complete] Best val loss: 0.009034 (Epoch 279)

--- STEP 1: FINE-TUNING (EXPERIMENTAL DATASET) ---
Experimental dataset size: 684
Split sizes: Train=536 | Val=85 | Test =63
Pretrained weights loaded successfully into fine-tuning architecture.
Epoch   50 | Train: 0.1111 | Val: 0.0323
Epoch  100 | Train: 0.0899 | Val: 0.0330
Epoch  150 | Train: 0.0811 | Val: 0.0300
Epoch  200 | Train: 0.0654 | Val: 0.0278
Epoch  250 | Train: 0.0594 | Val: 0.0243
Epoch  300 | Train: 0.0523 | Val: 0.0257
Epoch  350 

In [5]:
# ============================================================================
# PCA - pretrain - Physics - Temp interpolation 
# ============================================================================
import argparse
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from rdkit import Chem
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA

def main(exp_data_path, pretrain_data_path, output_dir, variance):
    seed_everything(SEED)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    calculator = make_calculator()

    # ============================================================================
    # 1. STEP 0 — PRETRAIN ON COSMO/XTB DATASET WITH UNIQUE-SOLVENT PCA
    # ============================================================================
    print("\n--- STEP 0: PRETRAINING WITH UNIQUE-SOLVENT PCA (COSMO/XTB DATASET) ---")
    pretrain_df = pd.read_csv(pretrain_data_path)
    pretrain_df.columns = pretrain_df.columns.str.strip()
    pretrain_df = pretrain_df.dropna(
        subset=["SMILES_1", "Temperature", "comp_1", "LogS"]
    ).reset_index(drop=True)

    # Compute raw Mordred descriptors for pretrain dataset
    unique_smiles_pre = pretrain_df["SMILES_1"].unique()
    pre_lookup = {
        smi: smiles_to_mordred(smi, calculator) for smi in unique_smiles_pre
    }
    fp1_pre_raw = np.vstack([pre_lookup[s] for s in pretrain_df["SMILES_1"]])
    fp2_pre_raw = fp1_pre_raw.copy()  # Pure solvent placeholder

    x1_pretrain = (pretrain_df["comp_1"].values / 100.0).reshape(-1, 1)
    T_pretrain = pretrain_df["Temperature"].values.reshape(-1, 1)
    Y_pretrain = pretrain_df["LogS"].values.reshape(-1, 1)

    # Train/Val split for pretraining
    idx_all_pre = np.arange(len(fp1_pre_raw))
    idx_pre_train, idx_pre_val = train_test_split(
        idx_all_pre, test_size=0.2, random_state=SEED
    )

    # ── Unique-solvent PCA, placed RIGHT AFTER fp generation ───────────────
    # compact_descriptors_unique_solvents expects a df with both SMILES columns.
    # The pretrain frame only has SMILES_1 (pure solvent), so we mirror it into
    # SMILES_2 in a copy so the unique-SMILES extraction sees each chemical once.
    pretrain_df_for_pca = pretrain_df.copy()
    pretrain_df_for_pca["SMILES_2"] = pretrain_df_for_pca["SMILES_1"]

    fp1_pre, fp2_pre, standardizer, pca = compact_descriptors_unique_solvents(
        pretrain_df_for_pca, fp1_pre_raw, fp2_pre_raw,
        idx_pre_train, variance=variance
    )
    # ───────────────────────────────────────────────────────────────────────

    print(f"Pretrain dataset size: {len(pretrain_df)}")
    print(f"PCA components: {pca.n_components_} | "
          f"Explained variance: {pca.explained_variance_ratio_.sum():.6f}")

    # Create PyTorch DataLoaders for pretraining
    split_pre = lambda a: (a[idx_pre_train], a[idx_pre_val])
    fp1_pre_tr, fp1_pre_va = split_pre(fp1_pre)
    fp2_pre_tr, fp2_pre_va = split_pre(fp2_pre)
    x_pre_tr, x_pre_va = split_pre(x1_pretrain)
    t_pre_tr, t_pre_va = split_pre(T_pretrain)
    y_pre_tr, y_pre_va = split_pre(Y_pretrain)

    train_loader_pre, val_loader_pre, _ = make_loaders(
        [fp1_pre_tr, fp2_pre_tr, x_pre_tr, t_pre_tr, y_pre_tr],
        [fp1_pre_va, fp2_pre_va, x_pre_va, t_pre_va, y_pre_va],
        [fp1_pre_va, fp2_pre_va, x_pre_va, t_pre_va, y_pre_va],
    )

    # Initialize and train pretraining model
    pretrain_model = WeightedSumFingerprintModel(n_desc=fp1_pre.shape[1]).to(DEVICE)
    print(f"Trainable parameters: {pretrain_model.count_parameters()}")

    forward = lambda m, batch: m(batch[0], batch[1], batch[2], batch[3])
    best_val_pre, _, _, epochs_pre = train_model(
        pretrain_model, train_loader_pre, val_loader_pre, forward
    )
    print(f"[Pretrain Complete] Best val loss: {best_val_pre:.6f} (Epoch {epochs_pre})")

    # Save pretrain model checkpoint and PCA transformer
    torch.save({
        "model_state_dict": pretrain_model.state_dict(),
        "model_config": {"n_desc": fp1_pre.shape[1]},
        "seed": SEED,
    }, output_dir / "pretrained_pca_unique_solvents_cosmo_model.pt")
    with (output_dir / "pretrained_pca_unique_solvents_cosmo_preprocessor.pkl").open("wb") as handle:
        pickle.dump({
            "fp_standardizer": standardizer,
            "pca": pca,
            "pca_variance_target": variance,
        }, handle)

    # ============================================================================
    # 2. STEP 1 — FINE-TUNE ON EXPERIMENTAL DATA (REUSE PRETRAINED PCA SPACE)
    # ============================================================================
    print("\n--- STEP 1: FINE-TUNING (EXPERIMENTAL DATASET) ---")
    df, x1, temperature, target = load_raw_data(exp_data_path)
    idx_train, idx_val, idx_test, sig, keys = temperature_holdout_split(df, mode="interpolation")

    print(f"Experimental dataset size: {len(df)}")
    print(f"Split sizes: Train={len(idx_train)} | Val={len(idx_val)} | "
          f"Test ={len(idx_test)}")

    fp1_exp_raw, fp2_exp_raw, descriptor_lookup = compute_fingerprints(df, calculator)

    # Transform experimental features using the PRETRAINED standardizer + PCA space
    fp1_exp_std = standardizer.transform(fp1_exp_raw)
    fp2_exp_std = standardizer.transform(fp2_exp_raw)
    fp1_exp = pca.transform(fp1_exp_std)
    fp2_exp = pca.transform(fp2_exp_std)

    split_exp = lambda a: (a[idx_train], a[idx_val], a[idx_test])
    fp1_tr, fp1_va, fp1_te = split_exp(fp1_exp)
    fp2_tr, fp2_va, fp2_te = split_exp(fp2_exp)
    x_tr, x_va, x_te = split_exp(x1)
    t_tr, t_va, t_te = split_exp(temperature)
    y_tr, y_va, y_te = split_exp(target)

    train_loader, val_loader, test_loader = make_loaders(
        [fp1_tr, fp2_tr, x_tr, t_tr, y_tr],
        [fp1_va, fp2_va, x_va, t_va, y_va],
        [fp1_te, fp2_te, x_te, t_te, y_te],
    )

    # Load pretrained weights into fine-tuning model
    finetune_model = WeightedSumFingerprintModel(n_desc=fp1_exp.shape[1]).to(DEVICE)
    checkpoint = torch.load(output_dir / "pretrained_pca_unique_solvents_cosmo_model.pt")
    finetune_model.load_state_dict(checkpoint["model_state_dict"])
    print("Pretrained weights loaded successfully into fine-tuning architecture.")

    best_val_ft, _, _, epochs_ft = train_model(
        finetune_model, train_loader, val_loader, forward
    )
    print(f"[Fine-tune Complete] Best val loss: {best_val_ft:.6f} (Epoch {epochs_ft})")

    # Evaluate metrics across all splits
    y_true_train, y_pred_train = predict_model(finetune_model, train_loader, forward)
    y_true_val, y_pred_val = predict_model(finetune_model, val_loader, forward)
    y_true_test, y_pred_test = predict_model(finetune_model, test_loader, forward)

    m_train = metrics(y_true_train, y_pred_train)
    m_val = metrics(y_true_val, y_pred_val)
    m_test = metrics(y_true_test, y_pred_test)

    print("\n--- FINE-TUNED METRICS (UNIQUE-SOLVENT PCA) ---")
    print(f"Train | R²={m_train['r2']:.4f} RMSE={m_train['rmse']:.4f} MAE={m_train['mae']:.4f}")
    print(f"Val   | R²={m_val['r2']:.4f} RMSE={m_val['rmse']:.4f} MAE={m_val['mae']:.4f}")
    print(f"Test  | R²={m_test['r2']:.4f} RMSE={m_test['rmse']:.4f} MAE={m_test['mae']:.4f}")

    plot_parity(
        {"Train": (y_true_train, y_pred_train), "Val": (y_true_val, y_pred_val), "Test": (y_true_test, y_pred_test)},
        output_dir / "Final_Model_Temp_Interpolation.png",
        title="Clozapine LogS Prediction Parity Plot"
    )


parser = argparse.ArgumentParser()
parser.add_argument("--exp-data", default="Clozapine_dataset.csv")
parser.add_argument("--pretrain-data", default="combined_file_XTB.csv")
parser.add_argument("--output-dir", default="Results")
parser.add_argument("--variance", type=float, default=0.95)
args = parser.parse_args([])

main(args.exp_data, args.pretrain_data, args.output_dir, args.variance)


--- STEP 0: PRETRAINING WITH UNIQUE-SOLVENT PCA (COSMO/XTB DATASET) ---
[TrainOnlyStandardizer] Descriptors: 681 total | 191 dropped (NaN) | 178 dropped (constant) | 312 remaining
Pretrain dataset size: 6000
PCA components: 61 | Explained variance: 0.950345
Trainable parameters: 6882
Epoch   50 | Train: 0.0285 | Val: 0.0153
Epoch  100 | Train: 0.0199 | Val: 0.0106
Epoch  150 | Train: 0.0175 | Val: 0.0096
Epoch  200 | Train: 0.0172 | Val: 0.0105
Epoch  250 | Train: 0.0174 | Val: 0.0106
[Pretrain Complete] Best val loss: 0.009034 (Epoch 279)

--- STEP 1: FINE-TUNING (EXPERIMENTAL DATASET) ---
Experimental dataset size: 684
Split sizes: Train=530 | Val=77 | Test =77
Pretrained weights loaded successfully into fine-tuning architecture.
Epoch   50 | Train: 0.0973 | Val: 0.0353
Epoch  100 | Train: 0.0837 | Val: 0.0309
Epoch  150 | Train: 0.0751 | Val: 0.0362
Epoch  200 | Train: 0.0609 | Val: 0.0366
Epoch  250 | Train: 0.0571 | Val: 0.0121
[Fine-tune Complete] Best val loss: 0.003712 (Epoch 

In [4]:
# ============================================================================
# PCA - pretrain - Physics - Temp Extrapolation 
# ============================================================================
import argparse
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from rdkit import Chem
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA

def main(exp_data_path, pretrain_data_path, output_dir, variance):
    seed_everything(SEED)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    calculator = make_calculator()

    # ============================================================================
    # 1. STEP 0 — PRETRAIN ON COSMO/XTB DATASET WITH UNIQUE-SOLVENT PCA
    # ============================================================================
    print("\n--- STEP 0: PRETRAINING WITH UNIQUE-SOLVENT PCA (COSMO/XTB DATASET) ---")
    pretrain_df = pd.read_csv(pretrain_data_path)
    pretrain_df.columns = pretrain_df.columns.str.strip()
    pretrain_df = pretrain_df.dropna(
        subset=["SMILES_1", "Temperature", "comp_1", "LogS"]
    ).reset_index(drop=True)

    # Compute raw Mordred descriptors for pretrain dataset
    unique_smiles_pre = pretrain_df["SMILES_1"].unique()
    pre_lookup = {
        smi: smiles_to_mordred(smi, calculator) for smi in unique_smiles_pre
    }
    fp1_pre_raw = np.vstack([pre_lookup[s] for s in pretrain_df["SMILES_1"]])
    fp2_pre_raw = fp1_pre_raw.copy()  # Pure solvent placeholder

    x1_pretrain = (pretrain_df["comp_1"].values / 100.0).reshape(-1, 1)
    T_pretrain = pretrain_df["Temperature"].values.reshape(-1, 1)
    Y_pretrain = pretrain_df["LogS"].values.reshape(-1, 1)

    # Train/Val split for pretraining
    idx_all_pre = np.arange(len(fp1_pre_raw))
    idx_pre_train, idx_pre_val = train_test_split(
        idx_all_pre, test_size=0.2, random_state=SEED
    )

    # ── Unique-solvent PCA, placed RIGHT AFTER fp generation ───────────────
    # compact_descriptors_unique_solvents expects a df with both SMILES columns.
    # The pretrain frame only has SMILES_1 (pure solvent), so we mirror it into
    # SMILES_2 in a copy so the unique-SMILES extraction sees each chemical once.
    pretrain_df_for_pca = pretrain_df.copy()
    pretrain_df_for_pca["SMILES_2"] = pretrain_df_for_pca["SMILES_1"]

    fp1_pre, fp2_pre, standardizer, pca = compact_descriptors_unique_solvents(
        pretrain_df_for_pca, fp1_pre_raw, fp2_pre_raw,
        idx_pre_train, variance=variance
    )
    # ───────────────────────────────────────────────────────────────────────

    print(f"Pretrain dataset size: {len(pretrain_df)}")
    print(f"PCA components: {pca.n_components_} | "
          f"Explained variance: {pca.explained_variance_ratio_.sum():.6f}")

    # Create PyTorch DataLoaders for pretraining
    split_pre = lambda a: (a[idx_pre_train], a[idx_pre_val])
    fp1_pre_tr, fp1_pre_va = split_pre(fp1_pre)
    fp2_pre_tr, fp2_pre_va = split_pre(fp2_pre)
    x_pre_tr, x_pre_va = split_pre(x1_pretrain)
    t_pre_tr, t_pre_va = split_pre(T_pretrain)
    y_pre_tr, y_pre_va = split_pre(Y_pretrain)

    train_loader_pre, val_loader_pre, _ = make_loaders(
        [fp1_pre_tr, fp2_pre_tr, x_pre_tr, t_pre_tr, y_pre_tr],
        [fp1_pre_va, fp2_pre_va, x_pre_va, t_pre_va, y_pre_va],
        [fp1_pre_va, fp2_pre_va, x_pre_va, t_pre_va, y_pre_va],
    )

    # Initialize and train pretraining model
    pretrain_model = WeightedSumFingerprintModel(n_desc=fp1_pre.shape[1]).to(DEVICE)
    print(f"Trainable parameters: {pretrain_model.count_parameters()}")

    forward = lambda m, batch: m(batch[0], batch[1], batch[2], batch[3])
    best_val_pre, _, _, epochs_pre = train_model(
        pretrain_model, train_loader_pre, val_loader_pre, forward
    )
    print(f"[Pretrain Complete] Best val loss: {best_val_pre:.6f} (Epoch {epochs_pre})")

    # Save pretrain model checkpoint and PCA transformer
    torch.save({
        "model_state_dict": pretrain_model.state_dict(),
        "model_config": {"n_desc": fp1_pre.shape[1]},
        "seed": SEED,
    }, output_dir / "pretrained_pca_unique_solvents_cosmo_model.pt")
    with (output_dir / "pretrained_pca_unique_solvents_cosmo_preprocessor.pkl").open("wb") as handle:
        pickle.dump({
            "fp_standardizer": standardizer,
            "pca": pca,
            "pca_variance_target": variance,
        }, handle)

    # ============================================================================
    # 2. STEP 1 — FINE-TUNE ON EXPERIMENTAL DATA (REUSE PRETRAINED PCA SPACE)
    # ============================================================================
    print("\n--- STEP 1: FINE-TUNING (EXPERIMENTAL DATASET) ---")
    df, x1, temperature, target = load_raw_data(exp_data_path)
    idx_train, idx_val, idx_test, sig, keys = temperature_holdout_split(df, mode="extrapolation")

    print(f"Experimental dataset size: {len(df)}")
    print(f"Split sizes: Train={len(idx_train)} | Val={len(idx_val)} | "
          f"Test ={len(idx_test)}")

    fp1_exp_raw, fp2_exp_raw, descriptor_lookup = compute_fingerprints(df, calculator)

    # Transform experimental features using the PRETRAINED standardizer + PCA space
    fp1_exp_std = standardizer.transform(fp1_exp_raw)
    fp2_exp_std = standardizer.transform(fp2_exp_raw)
    fp1_exp = pca.transform(fp1_exp_std)
    fp2_exp = pca.transform(fp2_exp_std)

    split_exp = lambda a: (a[idx_train], a[idx_val], a[idx_test])
    fp1_tr, fp1_va, fp1_te = split_exp(fp1_exp)
    fp2_tr, fp2_va, fp2_te = split_exp(fp2_exp)
    x_tr, x_va, x_te = split_exp(x1)
    t_tr, t_va, t_te = split_exp(temperature)
    y_tr, y_va, y_te = split_exp(target)

    train_loader, val_loader, test_loader = make_loaders(
        [fp1_tr, fp2_tr, x_tr, t_tr, y_tr],
        [fp1_va, fp2_va, x_va, t_va, y_va],
        [fp1_te, fp2_te, x_te, t_te, y_te],
    )

    # Load pretrained weights into fine-tuning model
    finetune_model = WeightedSumFingerprintModel(n_desc=fp1_exp.shape[1]).to(DEVICE)
    checkpoint = torch.load(output_dir / "pretrained_pca_unique_solvents_cosmo_model.pt")
    finetune_model.load_state_dict(checkpoint["model_state_dict"])
    print("Pretrained weights loaded successfully into fine-tuning architecture.")

    best_val_ft, _, _, epochs_ft = train_model(
        finetune_model, train_loader, val_loader, forward
    )
    print(f"[Fine-tune Complete] Best val loss: {best_val_ft:.6f} (Epoch {epochs_ft})")

    # Evaluate metrics across all splits
    y_true_train, y_pred_train = predict_model(finetune_model, train_loader, forward)
    y_true_val, y_pred_val = predict_model(finetune_model, val_loader, forward)
    y_true_test, y_pred_test = predict_model(finetune_model, test_loader, forward)

    m_train = metrics(y_true_train, y_pred_train)
    m_val = metrics(y_true_val, y_pred_val)
    m_test = metrics(y_true_test, y_pred_test)

    print("\n--- FINE-TUNED METRICS (UNIQUE-SOLVENT PCA) ---")
    print(f"Train | R²={m_train['r2']:.4f} RMSE={m_train['rmse']:.4f} MAE={m_train['mae']:.4f}")
    print(f"Val   | R²={m_val['r2']:.4f} RMSE={m_val['rmse']:.4f} MAE={m_val['mae']:.4f}")
    print(f"Test  | R²={m_test['r2']:.4f} RMSE={m_test['rmse']:.4f} MAE={m_test['mae']:.4f}")

    plot_parity(
        {"Train": (y_true_train, y_pred_train), "Val": (y_true_val, y_pred_val), "Test": (y_true_test, y_pred_test)},
        output_dir / "Final_Model_Temp_Extrapolation.png",
        title="Clozapine LogS Prediction Parity Plot"
    )


parser = argparse.ArgumentParser()
parser.add_argument("--exp-data", default="Clozapine_dataset.csv")
parser.add_argument("--pretrain-data", default="combined_file_XTB.csv")
parser.add_argument("--output-dir", default="Results")
parser.add_argument("--variance", type=float, default=0.95)
args = parser.parse_args([])

main(args.exp_data, args.pretrain_data, args.output_dir, args.variance)


--- STEP 0: PRETRAINING WITH UNIQUE-SOLVENT PCA (COSMO/XTB DATASET) ---
[TrainOnlyStandardizer] Descriptors: 681 total | 191 dropped (NaN) | 178 dropped (constant) | 312 remaining
Pretrain dataset size: 6000
PCA components: 61 | Explained variance: 0.950345
Trainable parameters: 6882
Epoch   50 | Train: 0.0285 | Val: 0.0153
Epoch  100 | Train: 0.0199 | Val: 0.0106
Epoch  150 | Train: 0.0175 | Val: 0.0096
Epoch  200 | Train: 0.0172 | Val: 0.0105
Epoch  250 | Train: 0.0174 | Val: 0.0106
[Pretrain Complete] Best val loss: 0.009034 (Epoch 279)

--- STEP 1: FINE-TUNING (EXPERIMENTAL DATASET) ---
Experimental dataset size: 684
Split sizes: Train=376 | Val=154 | Test =154
Pretrained weights loaded successfully into fine-tuning architecture.
Epoch   50 | Train: 0.0923 | Val: 0.0481
Epoch  100 | Train: 0.0981 | Val: 0.0424
Epoch  150 | Train: 0.0690 | Val: 0.0363
Epoch  200 | Train: 0.0770 | Val: 0.0379
Epoch  250 | Train: 0.0634 | Val: 0.0439
[Fine-tune Complete] Best val loss: 0.023309 (Epoc

In [3]:
# ============================================================================
# PCA - pretrain - Physics - Composition interpolation 
# ============================================================================
import argparse
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from rdkit import Chem
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA

def main(exp_data_path, pretrain_data_path, output_dir, variance):
    seed_everything(SEED)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    calculator = make_calculator()

    # ============================================================================
    # 1. STEP 0 — PRETRAIN ON COSMO/XTB DATASET WITH UNIQUE-SOLVENT PCA
    # ============================================================================
    print("\n--- STEP 0: PRETRAINING WITH UNIQUE-SOLVENT PCA (COSMO/XTB DATASET) ---")
    pretrain_df = pd.read_csv(pretrain_data_path)
    pretrain_df.columns = pretrain_df.columns.str.strip()
    pretrain_df = pretrain_df.dropna(
        subset=["SMILES_1", "Temperature", "comp_1", "LogS"]
    ).reset_index(drop=True)

    # Compute raw Mordred descriptors for pretrain dataset
    unique_smiles_pre = pretrain_df["SMILES_1"].unique()
    pre_lookup = {
        smi: smiles_to_mordred(smi, calculator) for smi in unique_smiles_pre
    }
    fp1_pre_raw = np.vstack([pre_lookup[s] for s in pretrain_df["SMILES_1"]])
    fp2_pre_raw = fp1_pre_raw.copy()  # Pure solvent placeholder

    x1_pretrain = (pretrain_df["comp_1"].values / 100.0).reshape(-1, 1)
    T_pretrain = pretrain_df["Temperature"].values.reshape(-1, 1)
    Y_pretrain = pretrain_df["LogS"].values.reshape(-1, 1)

    # Train/Val split for pretraining
    idx_all_pre = np.arange(len(fp1_pre_raw))
    idx_pre_train, idx_pre_val = train_test_split(
        idx_all_pre, test_size=0.2, random_state=SEED
    )

    # ── Unique-solvent PCA, placed RIGHT AFTER fp generation ───────────────
    # compact_descriptors_unique_solvents expects a df with both SMILES columns.
    # The pretrain frame only has SMILES_1 (pure solvent), so we mirror it into
    # SMILES_2 in a copy so the unique-SMILES extraction sees each chemical once.
    pretrain_df_for_pca = pretrain_df.copy()
    pretrain_df_for_pca["SMILES_2"] = pretrain_df_for_pca["SMILES_1"]

    fp1_pre, fp2_pre, standardizer, pca = compact_descriptors_unique_solvents(
        pretrain_df_for_pca, fp1_pre_raw, fp2_pre_raw,
        idx_pre_train, variance=variance
    )
    # ───────────────────────────────────────────────────────────────────────

    print(f"Pretrain dataset size: {len(pretrain_df)}")
    print(f"PCA components: {pca.n_components_} | "
          f"Explained variance: {pca.explained_variance_ratio_.sum():.6f}")

    # Create PyTorch DataLoaders for pretraining
    split_pre = lambda a: (a[idx_pre_train], a[idx_pre_val])
    fp1_pre_tr, fp1_pre_va = split_pre(fp1_pre)
    fp2_pre_tr, fp2_pre_va = split_pre(fp2_pre)
    x_pre_tr, x_pre_va = split_pre(x1_pretrain)
    t_pre_tr, t_pre_va = split_pre(T_pretrain)
    y_pre_tr, y_pre_va = split_pre(Y_pretrain)

    train_loader_pre, val_loader_pre, _ = make_loaders(
        [fp1_pre_tr, fp2_pre_tr, x_pre_tr, t_pre_tr, y_pre_tr],
        [fp1_pre_va, fp2_pre_va, x_pre_va, t_pre_va, y_pre_va],
        [fp1_pre_va, fp2_pre_va, x_pre_va, t_pre_va, y_pre_va],
    )

    # Initialize and train pretraining model
    pretrain_model = WeightedSumFingerprintModel(n_desc=fp1_pre.shape[1]).to(DEVICE)
    print(f"Trainable parameters: {pretrain_model.count_parameters()}")

    forward = lambda m, batch: m(batch[0], batch[1], batch[2], batch[3])
    best_val_pre, _, _, epochs_pre = train_model(
        pretrain_model, train_loader_pre, val_loader_pre, forward
    )
    print(f"[Pretrain Complete] Best val loss: {best_val_pre:.6f} (Epoch {epochs_pre})")

    # Save pretrain model checkpoint and PCA transformer
    torch.save({
        "model_state_dict": pretrain_model.state_dict(),
        "model_config": {"n_desc": fp1_pre.shape[1]},
        "seed": SEED,
    }, output_dir / "pretrained_pca_unique_solvents_cosmo_model.pt")
    with (output_dir / "pretrained_pca_unique_solvents_cosmo_preprocessor.pkl").open("wb") as handle:
        pickle.dump({
            "fp_standardizer": standardizer,
            "pca": pca,
            "pca_variance_target": variance,
        }, handle)

    # ============================================================================
    # 2. STEP 1 — FINE-TUNE ON EXPERIMENTAL DATA (REUSE PRETRAINED PCA SPACE)
    # ============================================================================
    print("\n--- STEP 1: FINE-TUNING (EXPERIMENTAL DATASET) ---")
    df, x1, temperature, target = load_raw_data(exp_data_path)
    idx_train, idx_val, idx_test, sig, keys = composition_holdout_split(df, mode="interpolation")

    print(f"Experimental dataset size: {len(df)}")
    print(f"Split sizes: Train={len(idx_train)} | Val={len(idx_val)} | "
          f"Test ={len(idx_test)}")

    fp1_exp_raw, fp2_exp_raw, descriptor_lookup = compute_fingerprints(df, calculator)

    # Transform experimental features using the PRETRAINED standardizer + PCA space
    fp1_exp_std = standardizer.transform(fp1_exp_raw)
    fp2_exp_std = standardizer.transform(fp2_exp_raw)
    fp1_exp = pca.transform(fp1_exp_std)
    fp2_exp = pca.transform(fp2_exp_std)

    split_exp = lambda a: (a[idx_train], a[idx_val], a[idx_test])
    fp1_tr, fp1_va, fp1_te = split_exp(fp1_exp)
    fp2_tr, fp2_va, fp2_te = split_exp(fp2_exp)
    x_tr, x_va, x_te = split_exp(x1)
    t_tr, t_va, t_te = split_exp(temperature)
    y_tr, y_va, y_te = split_exp(target)

    train_loader, val_loader, test_loader = make_loaders(
        [fp1_tr, fp2_tr, x_tr, t_tr, y_tr],
        [fp1_va, fp2_va, x_va, t_va, y_va],
        [fp1_te, fp2_te, x_te, t_te, y_te],
    )

    # Load pretrained weights into fine-tuning model
    finetune_model = WeightedSumFingerprintModel(n_desc=fp1_exp.shape[1]).to(DEVICE)
    checkpoint = torch.load(output_dir / "pretrained_pca_unique_solvents_cosmo_model.pt")
    finetune_model.load_state_dict(checkpoint["model_state_dict"])
    print("Pretrained weights loaded successfully into fine-tuning architecture.")

    best_val_ft, _, _, epochs_ft = train_model(
        finetune_model, train_loader, val_loader, forward
    )
    print(f"[Fine-tune Complete] Best val loss: {best_val_ft:.6f} (Epoch {epochs_ft})")

    # Evaluate metrics across all splits
    y_true_train, y_pred_train = predict_model(finetune_model, train_loader, forward)
    y_true_val, y_pred_val = predict_model(finetune_model, val_loader, forward)
    y_true_test, y_pred_test = predict_model(finetune_model, test_loader, forward)

    m_train = metrics(y_true_train, y_pred_train)
    m_val = metrics(y_true_val, y_pred_val)
    m_test = metrics(y_true_test, y_pred_test)

    print("\n--- FINE-TUNED METRICS (UNIQUE-SOLVENT PCA) ---")
    print(f"Train | R²={m_train['r2']:.4f} RMSE={m_train['rmse']:.4f} MAE={m_train['mae']:.4f}")
    print(f"Val   | R²={m_val['r2']:.4f} RMSE={m_val['rmse']:.4f} MAE={m_val['mae']:.4f}")
    print(f"Test  | R²={m_test['r2']:.4f} RMSE={m_test['rmse']:.4f} MAE={m_test['mae']:.4f}")

    plot_parity(
        {"Train": (y_true_train, y_pred_train), "Val": (y_true_val, y_pred_val), "Test": (y_true_test, y_pred_test)},
        output_dir / "Final_Model_Composition_Interpolation.png",
        title="Clozapine LogS Prediction Parity Plot"
    )


parser = argparse.ArgumentParser()
parser.add_argument("--exp-data", default="Clozapine_dataset.csv")
parser.add_argument("--pretrain-data", default="combined_file_XTB.csv")
parser.add_argument("--output-dir", default="Results")
parser.add_argument("--variance", type=float, default=0.95)
args = parser.parse_args([])

main(args.exp_data, args.pretrain_data, args.output_dir, args.variance)


--- STEP 0: PRETRAINING WITH UNIQUE-SOLVENT PCA (COSMO/XTB DATASET) ---
[TrainOnlyStandardizer] Descriptors: 681 total | 191 dropped (NaN) | 178 dropped (constant) | 312 remaining
Pretrain dataset size: 6000
PCA components: 61 | Explained variance: 0.950345
Trainable parameters: 6882
Epoch   50 | Train: 0.0285 | Val: 0.0153
Epoch  100 | Train: 0.0199 | Val: 0.0106
Epoch  150 | Train: 0.0175 | Val: 0.0096
Epoch  200 | Train: 0.0172 | Val: 0.0105
Epoch  250 | Train: 0.0174 | Val: 0.0106
[Pretrain Complete] Best val loss: 0.009034 (Epoch 279)

--- STEP 1: FINE-TUNING (EXPERIMENTAL DATASET) ---
Experimental dataset size: 684
Split sizes: Train=474 | Val=140 | Test =70
Pretrained weights loaded successfully into fine-tuning architecture.
Epoch   50 | Train: 0.1104 | Val: 0.0330
Epoch  100 | Train: 0.0984 | Val: 0.0178
Epoch  150 | Train: 0.0802 | Val: 0.0191
Epoch  200 | Train: 0.0796 | Val: 0.0129
Epoch  250 | Train: 0.0633 | Val: 0.0120
[Fine-tune Complete] Best val loss: 0.009127 (Epoch

In [2]:
# ============================================================================
# PCA - pretrain - Physics - Composition extrapolation 
# ============================================================================
import argparse
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from rdkit import Chem
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA

def main(exp_data_path, pretrain_data_path, output_dir, variance):
    seed_everything(SEED)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    calculator = make_calculator()

    # ============================================================================
    # 1. STEP 0 — PRETRAIN ON COSMO/XTB DATASET WITH UNIQUE-SOLVENT PCA
    # ============================================================================
    print("\n--- STEP 0: PRETRAINING WITH UNIQUE-SOLVENT PCA (COSMO/XTB DATASET) ---")
    pretrain_df = pd.read_csv(pretrain_data_path)
    pretrain_df.columns = pretrain_df.columns.str.strip()
    pretrain_df = pretrain_df.dropna(
        subset=["SMILES_1", "Temperature", "comp_1", "LogS"]
    ).reset_index(drop=True)

    # Compute raw Mordred descriptors for pretrain dataset
    unique_smiles_pre = pretrain_df["SMILES_1"].unique()
    pre_lookup = {
        smi: smiles_to_mordred(smi, calculator) for smi in unique_smiles_pre
    }
    fp1_pre_raw = np.vstack([pre_lookup[s] for s in pretrain_df["SMILES_1"]])
    fp2_pre_raw = fp1_pre_raw.copy()  # Pure solvent placeholder

    x1_pretrain = (pretrain_df["comp_1"].values / 100.0).reshape(-1, 1)
    T_pretrain = pretrain_df["Temperature"].values.reshape(-1, 1)
    Y_pretrain = pretrain_df["LogS"].values.reshape(-1, 1)

    # Train/Val split for pretraining
    idx_all_pre = np.arange(len(fp1_pre_raw))
    idx_pre_train, idx_pre_val = train_test_split(
        idx_all_pre, test_size=0.2, random_state=SEED
    )

    # ── Unique-solvent PCA, placed RIGHT AFTER fp generation ───────────────
    # compact_descriptors_unique_solvents expects a df with both SMILES columns.
    # The pretrain frame only has SMILES_1 (pure solvent), so we mirror it into
    # SMILES_2 in a copy so the unique-SMILES extraction sees each chemical once.
    pretrain_df_for_pca = pretrain_df.copy()
    pretrain_df_for_pca["SMILES_2"] = pretrain_df_for_pca["SMILES_1"]

    fp1_pre, fp2_pre, standardizer, pca = compact_descriptors_unique_solvents(
        pretrain_df_for_pca, fp1_pre_raw, fp2_pre_raw,
        idx_pre_train, variance=variance
    )
    # ───────────────────────────────────────────────────────────────────────

    print(f"Pretrain dataset size: {len(pretrain_df)}")
    print(f"PCA components: {pca.n_components_} | "
          f"Explained variance: {pca.explained_variance_ratio_.sum():.6f}")

    # Create PyTorch DataLoaders for pretraining
    split_pre = lambda a: (a[idx_pre_train], a[idx_pre_val])
    fp1_pre_tr, fp1_pre_va = split_pre(fp1_pre)
    fp2_pre_tr, fp2_pre_va = split_pre(fp2_pre)
    x_pre_tr, x_pre_va = split_pre(x1_pretrain)
    t_pre_tr, t_pre_va = split_pre(T_pretrain)
    y_pre_tr, y_pre_va = split_pre(Y_pretrain)

    train_loader_pre, val_loader_pre, _ = make_loaders(
        [fp1_pre_tr, fp2_pre_tr, x_pre_tr, t_pre_tr, y_pre_tr],
        [fp1_pre_va, fp2_pre_va, x_pre_va, t_pre_va, y_pre_va],
        [fp1_pre_va, fp2_pre_va, x_pre_va, t_pre_va, y_pre_va],
    )

    # Initialize and train pretraining model
    pretrain_model = WeightedSumFingerprintModel(n_desc=fp1_pre.shape[1]).to(DEVICE)
    print(f"Trainable parameters: {pretrain_model.count_parameters()}")

    forward = lambda m, batch: m(batch[0], batch[1], batch[2], batch[3])
    best_val_pre, _, _, epochs_pre = train_model(
        pretrain_model, train_loader_pre, val_loader_pre, forward
    )
    print(f"[Pretrain Complete] Best val loss: {best_val_pre:.6f} (Epoch {epochs_pre})")

    # Save pretrain model checkpoint and PCA transformer
    torch.save({
        "model_state_dict": pretrain_model.state_dict(),
        "model_config": {"n_desc": fp1_pre.shape[1]},
        "seed": SEED,
    }, output_dir / "pretrained_pca_unique_solvents_cosmo_model.pt")
    with (output_dir / "pretrained_pca_unique_solvents_cosmo_preprocessor.pkl").open("wb") as handle:
        pickle.dump({
            "fp_standardizer": standardizer,
            "pca": pca,
            "pca_variance_target": variance,
        }, handle)

    # ============================================================================
    # 2. STEP 1 — FINE-TUNE ON EXPERIMENTAL DATA (REUSE PRETRAINED PCA SPACE)
    # ============================================================================
    print("\n--- STEP 1: FINE-TUNING (EXPERIMENTAL DATASET) ---")
    df, x1, temperature, target = load_raw_data(exp_data_path)
    idx_train, idx_val, idx_test, sig, keys = composition_holdout_split(df, mode="extrapolation")

    print(f"Experimental dataset size: {len(df)}")
    print(f"Split sizes: Train={len(idx_train)} | Val={len(idx_val)} | "
          f"Test ={len(idx_test)}")

    fp1_exp_raw, fp2_exp_raw, descriptor_lookup = compute_fingerprints(df, calculator)

    # Transform experimental features using the PRETRAINED standardizer + PCA space
    fp1_exp_std = standardizer.transform(fp1_exp_raw)
    fp2_exp_std = standardizer.transform(fp2_exp_raw)
    fp1_exp = pca.transform(fp1_exp_std)
    fp2_exp = pca.transform(fp2_exp_std)

    split_exp = lambda a: (a[idx_train], a[idx_val], a[idx_test])
    fp1_tr, fp1_va, fp1_te = split_exp(fp1_exp)
    fp2_tr, fp2_va, fp2_te = split_exp(fp2_exp)
    x_tr, x_va, x_te = split_exp(x1)
    t_tr, t_va, t_te = split_exp(temperature)
    y_tr, y_va, y_te = split_exp(target)

    train_loader, val_loader, test_loader = make_loaders(
        [fp1_tr, fp2_tr, x_tr, t_tr, y_tr],
        [fp1_va, fp2_va, x_va, t_va, y_va],
        [fp1_te, fp2_te, x_te, t_te, y_te],
    )

    # Load pretrained weights into fine-tuning model
    finetune_model = WeightedSumFingerprintModel(n_desc=fp1_exp.shape[1]).to(DEVICE)
    checkpoint = torch.load(output_dir / "pretrained_pca_unique_solvents_cosmo_model.pt")
    finetune_model.load_state_dict(checkpoint["model_state_dict"])
    print("Pretrained weights loaded successfully into fine-tuning architecture.")

    best_val_ft, _, _, epochs_ft = train_model(
        finetune_model, train_loader, val_loader, forward
    )
    print(f"[Fine-tune Complete] Best val loss: {best_val_ft:.6f} (Epoch {epochs_ft})")

    # Evaluate metrics across all splits
    y_true_train, y_pred_train = predict_model(finetune_model, train_loader, forward)
    y_true_val, y_pred_val = predict_model(finetune_model, val_loader, forward)
    y_true_test, y_pred_test = predict_model(finetune_model, test_loader, forward)

    m_train = metrics(y_true_train, y_pred_train)
    m_val = metrics(y_true_val, y_pred_val)
    m_test = metrics(y_true_test, y_pred_test)

    print("\n--- FINE-TUNED METRICS (UNIQUE-SOLVENT PCA) ---")
    print(f"Train | R²={m_train['r2']:.4f} RMSE={m_train['rmse']:.4f} MAE={m_train['mae']:.4f}")
    print(f"Val   | R²={m_val['r2']:.4f} RMSE={m_val['rmse']:.4f} MAE={m_val['mae']:.4f}")
    print(f"Test  | R²={m_test['r2']:.4f} RMSE={m_test['rmse']:.4f} MAE={m_test['mae']:.4f}")

    plot_parity(
        {"Train": (y_true_train, y_pred_train), "Val": (y_true_val, y_pred_val), "Test": (y_true_test, y_pred_test)},
        output_dir / "Final_Model_Composition_Extrapolation.png",
        title="Clozapine LogS Prediction Parity Plot"
    )


parser = argparse.ArgumentParser()
parser.add_argument("--exp-data", default="Clozapine_dataset.csv")
parser.add_argument("--pretrain-data", default="combined_file_XTB.csv")
parser.add_argument("--output-dir", default="Results")
parser.add_argument("--variance", type=float, default=0.95)
args = parser.parse_args([])

main(args.exp_data, args.pretrain_data, args.output_dir, args.variance)


--- STEP 0: PRETRAINING WITH UNIQUE-SOLVENT PCA (COSMO/XTB DATASET) ---
[TrainOnlyStandardizer] Descriptors: 681 total | 191 dropped (NaN) | 178 dropped (constant) | 312 remaining
Pretrain dataset size: 6000
PCA components: 61 | Explained variance: 0.950345
Trainable parameters: 6882
Epoch   50 | Train: 0.0285 | Val: 0.0153
Epoch  100 | Train: 0.0199 | Val: 0.0106
Epoch  150 | Train: 0.0175 | Val: 0.0096
Epoch  200 | Train: 0.0172 | Val: 0.0105
Epoch  250 | Train: 0.0174 | Val: 0.0106
[Pretrain Complete] Best val loss: 0.009034 (Epoch 279)

--- STEP 1: FINE-TUNING (EXPERIMENTAL DATASET) ---
Experimental dataset size: 684
Split sizes: Train=412 | Val=140 | Test =132
Pretrained weights loaded successfully into fine-tuning architecture.
Epoch   50 | Train: 0.0379 | Val: 0.1268
Epoch  100 | Train: 0.0340 | Val: 0.1079
Epoch  150 | Train: 0.0284 | Val: 0.0924
Epoch  200 | Train: 0.0244 | Val: 0.0835
Epoch  250 | Train: 0.0273 | Val: 0.0805
Epoch  300 | Train: 0.0242 | Val: 0.0755
[Fine-tun